# Arbitrage-Aware Probabilistic Multi-Step Forecasting of Implied Volatility Surfaces: Joint Generation of Surface-Return Trajectories Using Latent Diffusion

This notebook implements a probabilistic forecasting pipeline for SPX implied volatility surfaces and index returns. Daily option quotes are first mapped to a common surface representation. A surface autoencoder then provides a compact latent state, and a conditional diffusion model generates joint trajectories over the next 30 trading observations.

### Contents

| Section | Purpose |
| :-- | :-- |
| [1. Environment and storage](#environment) | Colab runtime, project paths, and dependencies |
| [2. Data and surface construction](#data) | WRDS quotes, smoothing, time-scaled volatility, and return features |
| [3. Datasets](#datasets) | Masks, normalisation, and forecast windows |
| [4. Surface autoencoder](#autoencoder) | Residual encoder, continuous decoder, and arbitrage penalties |
| [5. Latent trajectory diffusion](#diffusion) | Joint targets, transformer denoiser, training, and conditional mean gate |
| [6. Forecast archive](#forecast-archive) | DDPM/DDIM ensembles, decoding, and saved forecast objects |

### Reading and execution context

The paths and imports target **Google Colab with Google Drive**. WRDS access requires an account with the relevant OptionMetrics permissions.

The standard numerical libraries imported here are expected to be available in the runtime. GPU acceleration is used when available.

Several stages reuse files when their configured paths already exist. Those files form part of the run state: a stored checkpoint or forecast archive takes precedence over retraining or resampling.

---


<a id="environment"></a>
## 1. Environment and storage

This section establishes the working directory. Relative paths in later sections are resolved from
the project folder on Google Drive.


### 1.1 Mount Google Drive

Mount Drive before accessing the project directory. This makes downloaded data and model checkpoints available across Colab sessions.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

### 1.2 Project paths and saved artifacts

For a new project folder, the setup creates the data subdirectories and changes the working directory to `MyDrive/IVS`. The following path constants are reused throughout the notebook.

In [ ]:
path_main_folder = "IVS"

if not os.path.exists(f"/content/drive/MyDrive/{path_main_folder}"):
    os.makedirs(f"/content/drive/MyDrive/{path_main_folder}")
    os.makedirs(f"/content/drive/MyDrive/{path_main_folder}/data/raw")
    os.makedirs(f"/content/drive/MyDrive/{path_main_folder}/data/opDsCache")
    os.makedirs(f"/content/drive/MyDrive/{path_main_folder}/data/opDsProcessed")
    os.makedirs(f"/content/drive/MyDrive/{path_main_folder}/data/processed")
os.chdir(f"/content/drive/MyDrive/{path_main_folder}")

# Centralise paths so all later stages share the same saved artifacts.
PATH_RAW_OPTIONPRICES = "data/raw/SPX_OptionPrices"
PATH_RAW_SPXPRICES = "data/raw/SPX_Prices"
PATH_OPDS_CACHE = "data/opDsCache"
PATH_OPDS_PROCESSED = "data/opDsProcessed"
PATH_SPX_tIVS = "data/processed/SPX_timescaled_IVS"
PATH_SPX_RETURNS = "data/processed/SPX_returns_features"
PATH_APPENDIX = "vF"
PATH_FINAL_AE_MODEL = f"RAE_FINAL_{PATH_APPENDIX}.pt"
PATH_FINAL_DIFF_MODEL = f"DIFF_FINAL_{PATH_APPENDIX}.pt"
PATH_DUMP = f"diffusion_dump_{PATH_APPENDIX}.pt"

### 1.3 Core imports

Load the tabular and array libraries, progress display, and filesystem utilities.
Additional dependencies are introduced alongside the stage that uses them.

In [ ]:
import subprocess
import pandas as pd
import numpy as np
from tqdm import tqdm
import shutil

### 1.4 Smoothing source and checkpoint location

Clone the implementation repository if it is absent. `ods_path` points to its Operator Deep Smoothing package; the pretrained smoother is loaded from the same repository later in the data pipeline.

The original implementation and the corresponding paper can be found [here](https://github.com/rwicl/operator-deep-smoothing-for-implied-volatility), but there has been in an update in an underlying package, making it incompatible with our current code. Hence, we provide a copy of the old code version in our own repository.

In [ ]:
if not os.path.exists("implied-volatility-trajectories"):
    subprocess.run([
        "git", "clone",
        "https://github.com/DomBBB/implied-volatility-trajectories.git"
    ], check=True)

ods_path = os.path.abspath(
    "implied-volatility-trajectories/ICAIF26/"
    "operator-deep-smoothing"
)

<a id="data"></a>
## 2. Data and surface construction

The data pipeline has three steps: download SPX option quotes and index prices, evaluate the pretrained surface smoother, and assemble the surface and return
series consumed by the forecasting models. Annual caches allow the quote and smoothing stages to be reused.

### 2.1 WRDS source data

The source is OptionMetrics through WRDS. The notebook selects SPX using
`secid = 108105` and requests annual option tables for **2000–2025**.


#### WRDS client

Install and import the client used for the database queries.

In [ ]:
subprocess.run(["pip", "install", "wrds"], check=True)
import wrds

#### Download quotes and index closes

The connection is authenticated interactively through WRDS. Access requires a dedicated WRDS account with 2FA enabled via Duo app (.pgpass file does not need to be created).

Option quotes are stored one year at a time, retaining the quote date, expiry, strike, bid, offer,
call/put flag, and settlement flag. The index query retains dates and closing prices from 3 January 2000 onward. Each download is skipped when its CSV already exists.

In [ ]:
db = wrds.Connection()
SECID = "108105"
DATE = "2000-01-03"

# Cache option quotes by year.
for year in tqdm(range(2000, 2026)):
    path_option_prices = f"{PATH_RAW_OPTIONPRICES}_{year}.csv"
    if not os.path.exists(path_option_prices):
        df_year = db.raw_sql(f"SELECT * FROM optionm_all.opprcd{year} WHERE secid='{SECID}'")
        df_year = df_year[["date", "exdate", "strike_price", "best_bid", "best_offer", "cp_flag", "am_settlement"]].copy()
        df_year["date"] = pd.to_datetime(df_year["date"], format="%Y-%m-%d")
        df_year["exdate"] = pd.to_datetime(df_year["exdate"], format="%Y-%m-%d")
        df_year = df_year.sort_values(["date", "exdate", "strike_price", "cp_flag"]).reset_index(drop=True)
        df_year.to_csv(path_option_prices, index=False)
        del df_year
        print(f"Stored {path_option_prices} on Google Drive.")
    else:
        print(f"File {path_option_prices} already exists!")

# Keep the index-close series in one file.
path_spx_prices = f"{PATH_RAW_SPXPRICES}.csv"
if not os.path.exists(path_spx_prices):
    df_prices = db.raw_sql(f"SELECT * FROM optionm_all.secprd WHERE date >= '{DATE}' AND secid='{SECID}'")
    df_prices = df_prices[["date", "close"]].copy()
    df_prices["date"] = pd.to_datetime(df_prices["date"], format="%Y-%m-%d")
    df_prices = df_prices.sort_values("date").reset_index(drop=True)
    df_prices.to_csv(path_spx_prices, index=False)
    del df_prices
    print(f"Stored {path_spx_prices} on Google Drive.")
else:
    print(f"File {path_spx_prices} already exists!")

db.close()

### 2.2 Operator Deep Smoothing

The pretrained graph-based smoother converts each day's irregular quote set to surface values at the requested coordinates. This stage prepares inputs for the forecasting models; it does not train the smoother in this notebook.

#### Smoothing dependencies

Install the graph, option-pricing, and tensor-container dependencies, then add
the cloned smoothing package to Python's import path.

In [ ]:
subprocess.run(["pip", "install", "torch-geometric"], check=True)
subprocess.run([
    "pip", "install",
    "--no-deps",
    "py-lets-be-rational==1.0.1",
    "py-vollib==1.0.1",
    "py-vollib-vectorized==0.1.1",
    "simplejson",
], check=True)
subprocess.run(["pip", "install", "tensordict"], check=True)
import sys
sys.path.insert(0, ods_path)

#### Random state and device

Set the Python, NumPy, and PyTorch seeds and request deterministic cuDNN behaviour. These settings control the random streams used here; exact numerical agreement
can still depend on the runtime and hardware.

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)

torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Using {device} for inference')

#### Smoother interfaces

Import the WRDS dataset wrapper, graph dataset adapter, coordinate grid, checkpoint loader, and graph-construction utilities.

In [ ]:
from volatility_smoothing.utils.train import misc
from volatility_smoothing.utils.options_data import WRDSOptionsDataset
from volatility_smoothing.utils.train.dataset import GNOOptionsDataset
from op_ds.utils.grid import RectilinearGrid
from torch.utils.data import DataLoader
from volatility_smoothing.utils.train.edge_index import generate_edge_index
from tensordict import TensorDict

#### Evaluation coordinates

The requested maturities run from **one to twelve months**. The smoother uses
coordinates $r=\sqrt{\tau}$ and a scaled moneyness coordinate, computed here as
$z_{\mathrm{OPDS}}=m/r$. `lookup_grid` records the correspondence back to the
requested moneyness values. Only scaled coordinates in the specified
$[-1.5,\,0.5]$ interval are retained.

The resulting observed lattice is a subset of the rectangular maturity–moneyness
mesh used by the neural models.

In [ ]:
taus = np.array([365/12*1,
                 365/12*2,
                 365/12*3,
                 365/12*4,
                 365/12*5,
                 365/12*6,
                 365/12*7,
                 365/12*8,
                 365/12*9,
                 365/12*10,
                 365/12*11,
                 365/12*12]) / 365
rs = np.round(np.sqrt(taus), 4)

ms = np.array([-1.5, -1.4, -1.3, -1.2, -1.1,
               -1.0, -0.9, -0.8, -0.7, -0.6,
               -0.5, -0.4, -0.3, -0.2, -0.1,
                0.0,  0.1,  0.2,  0.3,  0.4,
                0.5])
# Translate requested moneyness into the smoother's scaled coordinates.
lookup_zs = np.round(ms[np.newaxis,:]/rs[:,np.newaxis], 4)
lookup_grid = pd.DataFrame(lookup_zs, index=rs, columns=ms)
unique_zs = np.unique(lookup_zs)
zs = unique_zs[(unique_zs >= -1.5) & (unique_zs <= 0.5)]

#### From one quote set to a graph input

`opds_collate_fn` assembles the observed `(r, z)` coordinates, requested output
grid, and graph edges for one date. The loader uses a batch size of one because
the quote set and graph vary by date.

In [ ]:
def opds_collate_fn(data_list):
    data = data_list[0]
    pos_x = torch.cat((data["r"], data["z"]), dim=1)
    grid = RectilinearGrid(r=torch.tensor(rs, dtype=torch.float32), z=torch.tensor(zs, dtype=torch.float32)).to(pos_x.device)
    pos_y = grid.flatten("channel_last")
    # Connect the irregular quote locations to the requested output grid.
    edge_index = generate_edge_index(pos_x, pos_y, subsample_size=35, radius=0.3)
    input = TensorDict({"x": data["implied_volatility"],
                        "pos_x": pos_x, "pos_y": pos_y,
                        "edge_index": edge_index})
    aux = {"grid": grid}
    return (data, input, aux)

#### Cache annual quote datasets

For an uncached year, the raw CSV is copied to temporary local storage and the
listed date–expiry exclusions are applied before the smoother's dataset wrapper
builds its cache. The environment variables point that wrapper to the current year's data and cache directory.

In [ ]:
for year in tqdm(range(2000, 2026)):

    path_temp_local = ""
    path_data_cached = f"{PATH_OPDS_CACHE}/{year}"

    if not os.path.exists(path_data_cached):
        path_temp_local = os.path.expanduser("~/temp")
        if os.path.exists(path_temp_local):
            shutil.rmtree(path_temp_local)
        os.makedirs(path_temp_local)
        src_path = f"{PATH_RAW_OPTIONPRICES}_{year}.csv"
        dst_path = path_temp_local + "/dataset.csv"
        subprocess.run(["cp", src_path, dst_path], check=True)
        print(f"{dst_path} copied to root.")
        # Retain the specified quote exclusions before constructing this year's cache.
        to_remove = [("2000-09-15", "2000-09-16"), # encoding problem in WRDS? 1d quotes & other missing data

                     ("2001-01-19", "2001-01-20"),
                     ("2001-06-15", "2001-06-16"),

                     ("2002-01-18", "2002-01-19"),
                     ("2002-06-17", "2004-06-19"),
                     ("2002-06-18", "2004-06-19"),
                     ("2002-06-19", "2004-06-19"),
                     ("2002-06-20", "2004-06-19"),
                     ("2002-06-21", "2004-06-19"),
                     ("2002-06-24", "2004-06-19"),
                     ("2002-08-16", "2002-08-17"),

                     ("2003-01-17", "2003-01-18"),
                     ("2003-06-19", "2005-06-18"),
                     ("2003-06-20", "2005-06-18"),
                     ("2003-06-23", "2005-06-18"),
                     ("2003-09-19", "2003-09-20"),

                     ("2004-03-19", "2004-03-20"),
                     ("2004-04-16", "2004-04-17"),

                     ("2006-11-17", "2006-11-18"),

                     ("2007-06-15", "2007-06-16"),

        ]
        remove_this = False
        for _r in to_remove:
            _d, _ex = _r
            if _d.startswith(str(year)):
                remove_this = True
                break
        if remove_this:
            print("Removal: ")
            df = pd.read_csv(dst_path)
            df_size_before = len(df)
            removed_string = "Removed: "
            removed_flag = False
            for _r in to_remove:
                _d, _ex = _r
                if _d.startswith(str(year)):
                    mask = (df["date"] == _d) & (df["exdate"] == _ex)
                    df = df.loc[~mask]
                    removed_string += f"({_d}), "
                    removed_flag = True
            assert removed_flag
            df.to_csv(dst_path, index=False)
            removed_string += f"for a total of {df_size_before-len(df)} removals."
            print(removed_string)

    os.environ["OPDS_WRDS_DATA_DIR"] = path_temp_local
    os.environ["OPDS_CACHE_DIR"] = path_data_cached
    spx_dataset = WRDSOptionsDataset()
    print(f"Dataset {year} cached.")

#### Load the pretrained smoother

Restore the supplied graph neural operator checkpoint, move it to the selected
device, and use evaluation mode for surface inference.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gno, _ = misc.load_checkpoint("implied-volatility-trajectories/ICAIF26/operator-deep-smoothing/volatility_smoothing/train/store/9448705/checkpoints/checkpoint_final.pt", device=device)
gno.to(device)
gno = gno.eval()

#### Smooth and export annual surfaces

Evaluate the smoother date by date, map output coordinates back through the
lookup table, and save matched `(date, maturity, moneyness, IV)` rows. An annual
output file is reused when it already exists.

In [ ]:
for year in tqdm(range(2000, 2026)):

    path_opds_processed = f"{PATH_OPDS_PROCESSED}/{year}.csv"
    if not os.path.exists(path_opds_processed):

        os.environ["OPDS_CACHE_DIR"] = f"{PATH_OPDS_CACHE}/{year}"
        spx_dataset = WRDSOptionsDataset()
        spx_gno_dataset = GNOOptionsDataset(spx_dataset, r_lim=(0.01, 1.), z_lim=(-1.5, 0.5))

        dataloader = DataLoader(spx_gno_dataset, batch_size=1, shuffle=False, collate_fn=opds_collate_fn)

        extract_data = {"date": [],
                "ttm": [],
                "moneyness": [],
                "iv": []}

        for data, input, aux in dataloader:

            data = data.to(device)
            input = input.to(device)
            grid = aux["grid"].to(device)

            with torch.no_grad():
                output = gno(**input)
            iv_predict, iv_y = output
            iv_surface = iv_y.view(grid.size())

            # Export only output coordinates that match the requested lookup lattice.
            for g_r_idx, g_r in enumerate(grid[0]):
                r = np.round(float(g_r[0]), 4)
                tau = taus[g_r_idx]
                for g_z_idx, g_z in enumerate(grid[1][g_r_idx]):
                    z = np.round(float(g_z), 4)
                    matches = lookup_grid.iloc[g_r_idx][lookup_grid.iloc[g_r_idx] == z]
                    if len(matches) == 0:
                        continue
                    elif len(matches) == 1:
                        m = matches.index.tolist()[0]
                        iv = iv_surface[g_r_idx][g_z_idx]
                        extract_data["date"].append(data["quote_datetime"].date())
                        extract_data["ttm"].append(tau)
                        extract_data["moneyness"].append(m)
                        extract_data["iv"].append(float(iv))
                    else:
                        raise ValueError
        df_extract_data = pd.DataFrame(extract_data)
        df_extract_data.to_csv(path_opds_processed, index=False)
        del extract_data
        del df_extract_data
        print(f"Stored {path_opds_processed} on Google Drive.")
    else:
        print(f"File {path_opds_processed} already exists!")

### 2.3 Consolidated forecasting inputs

The annual smoothed surfaces are combined into a single time series. Index closes are processed separately into log returns and trailing-window features;
the two series are joined by date in Section 3.

#### Time-scaled implied volatility

Write ordinary implied volatility as $\sigma_t(m,\tau)$ and define

$$
\Xi_t(m,\tau)=\sqrt{\tau}\,\sigma_t(m,\tau).
$$

The CSV column `t-iv` stores $\Xi$. Before saving it, the code checks for
constant maturity slices and applies its specified repair: interpolate
**total implied variance**, $w=\tau\sigma^2=\Xi^2$, between the neighbouring
maturities, then convert back to ordinary IV. The assertions identify the
particular maturity configuration expected by this repair.


In [ ]:
path_spx_tivs = f"{PATH_SPX_tIVS}.csv"

if not os.path.exists(path_spx_tivs):
    all_dfs = []
    for year in tqdm(range(2000, 2026)):
        path_data_processed = f"{PATH_OPDS_PROCESSED}/{year}.csv"
        df = pd.read_csv(path_data_processed)
        df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
        df = df[["date", "ttm", "moneyness", "iv"]]
        print(year)
        g = df.groupby(["date", "ttm"])["iv"]
        stats = g.agg(n="size", min="min", max="max")
        stats["flat"] = (stats["max"] - stats["min"]) == 0
        flat_pairs = stats[stats["flat"]].reset_index()
        if len(flat_pairs) > 0:
            print(f"Flat pairs: {len(flat_pairs)}")
            replacement_dfs = []
            seen = []
            for d, t in flat_pairs[["date", "ttm"]].itertuples(index=False):
                print(f"#### Correcting {d} - {t}")
                assert d not in seen
                seen.append(d)
                df_d = df[df["date"] == d]
                ttms = sorted(df_d["ttm"].unique())
                df_t = df_d[df_d["ttm"] == t]
                down_idx = ttms.index(t) - 1
                down_ttm = ttms[down_idx]
                up_idx = ttms.index(t) + 1
                up_ttm = ttms[up_idx]
                assert down_ttm == 0.25
                assert up_ttm == 0.4166666666666667
                df_down = df_d[df_d["ttm"] == ttms[down_idx]][["moneyness", "iv"]]
                df_down.rename(columns={"iv": "down"}, inplace=True)
                df_final = df_t.merge(df_down, on="moneyness", how="left")
                df_up = df_d[df_d["ttm"] == ttms[up_idx]][["moneyness", "iv"]]
                df_up.rename(columns={"iv": "up"}, inplace=True)
                df_final = df_final.merge(df_up, on="moneyness", how="left")
                # Repair in total-variance units before converting back to ordinary IV.
                alpha = (t-down_ttm) / (up_ttm-down_ttm)
                w_m = (df_final["down"]**2) * down_ttm
                w_p = (df_final["up"]**2) * up_ttm
                w_0 = np.where(np.isnan(w_m), w_p,
                               np.where(np.isnan(w_p), w_m,
                               (1-alpha)*w_m + alpha*w_p))
                df_final["iv_replaced"] = np.sqrt(w_0 / t)
                replacement_dfs.append(df_final)
            replacement_df = pd.concat(replacement_dfs, ignore_index=True)
            assert len(replacement_df) == len(flat_pairs) * 11
            repl = replacement_df[["date", "ttm", "moneyness", "iv_replaced"]].copy()
            df2 = df.merge(repl, on=["date", "ttm", "moneyness"], how="left")
            df2["iv"] = np.where(df2["iv_replaced"].notna(), df2["iv_replaced"], df2["iv"])
            df2 = df2.drop(columns=["iv_replaced"])
            df = df2
            print("Successfully replaced.")
        # Xi = sqrt(tau) * IV is the surface quantity modelled downstream.
        df["t-iv"] = np.sqrt(df["ttm"]) * df["iv"]
        df = df[["date", "ttm", "moneyness", "t-iv"]]
        all_dfs.append(df)
        print("-------------------------------")
    df_SPX_tIVS = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    df_SPX_tIVS = df_SPX_tIVS.sort_values("date").reset_index(drop=True)
    df_SPX_tIVS.to_csv(path_spx_tivs, index=False)
    del df_SPX_tIVS
    print(f"Stored {path_spx_tivs} on Google Drive.")
else:
    print(f"File {path_spx_tivs} already exists!")

#### Index returns and trailing features

Daily log returns are computed from consecutive index closes. The conditioning
features summarise return sign and magnitude, 5- and 21-observation cumulative
returns, realised volatility, downside semivolatility, skewness, extremes, and a
volatility-of-volatility proxy. All rolling windows end on the current date.

Realised-volatility features use an annualisation factor of 252. Initial rows
without a complete feature history are removed when the surface and return
tables are joined.

In [ ]:
path_spx_ret = f"{PATH_SPX_RETURNS}.csv"

if not os.path.exists(path_spx_ret):
    path_prices_raw = f"{PATH_RAW_SPXPRICES}.csv"
    df_SPX_ret = pd.read_csv(path_prices_raw)
    df_SPX_ret["date"] = pd.to_datetime(df_SPX_ret["date"], format="%Y-%m-%d")
    df_SPX_ret = df_SPX_ret.sort_values("date").reset_index(drop=True)

    # Daily log return
    df_SPX_ret["log_ret"] = np.log(df_SPX_ret["close"] / df_SPX_ret["close"].shift(1))
    r = df_SPX_ret["log_ret"].copy()

    # Negative-return indicator
    df_SPX_ret["ret_neg_ind"] = (r < 0.0).astype(np.int8)

    # Squared return
    df_SPX_ret["ret_sq"] = r.pow(2)

    # Min & Max over medium horizon
    df_SPX_ret["min_21d"] = r.rolling(window=21, min_periods=21).min()
    df_SPX_ret["max_21d"] = r.rolling(window=21, min_periods=21).max()

    # Cumulative returns over short / medium horizon
    df_SPX_ret["cumret_5d"] = r.rolling(window=5, min_periods=5).sum()
    df_SPX_ret["cumret_21d"] = r.rolling(window=21, min_periods=21).sum()

    # Rolling realized volatility:
    # RV_w = sqrt(252 * mean(r^2 over last w days))
    df_SPX_ret["rv_5d"] = np.sqrt(252.0 * r.pow(2).rolling(window=5, min_periods=5).mean())
    df_SPX_ret["rv_21d"] = np.sqrt(252.0 * r.pow(2).rolling(window=21, min_periods=21).mean())

    # Rolling downside semivolatility:
    # DSV_w = sqrt(252 * mean(min(r,0)^2 over last w days))
    r_neg = r.clip(upper=0.0)
    df_SPX_ret["dsv_21d"] = np.sqrt(252.0 * r_neg.pow(2).rolling(window=21, min_periods=21).mean())

    # Rolling return skewness over 21 days
    df_SPX_ret["ret_skew_21d"] = r.rolling(window=21, min_periods=21).skew()

    # Vol-of-vol proxy:
    # std of the past 21 observations of 5d RV
    df_SPX_ret["vov_21d_on_rv5"] = (
        df_SPX_ret["rv_5d"]
        .rolling(window=21, min_periods=21)
        .std(ddof=1)
    )

    df_SPX_ret = df_SPX_ret[
        [
            "date",
            "log_ret",
            "ret_neg_ind",
            "ret_sq",
            "cumret_5d",
            "cumret_21d",
            "rv_5d",
            "rv_21d",
            "dsv_21d",
            "ret_skew_21d",
            "vov_21d_on_rv5",
            "min_21d", "max_21d"
        ]
    ]

    # we lose first 25 observations
    df_SPX_ret.to_csv(path_spx_ret, index=False)
    del df_SPX_ret
    print(f"Stored {path_spx_ret} on Google Drive.")
else:
    print(f"File {path_spx_ret} already exists!")

<a id="datasets"></a>
## 3. Datasets

This section defines the representation shared by the autoencoder and diffusion
model. A single dataset class supports both daily surface reconstruction and
forecast-origin windows, keeping date alignment and the observation mask in one
place.

### 3.1 Modelling runtime

Load the modelling dependencies, initialise the random streams, and select the
device. Mixed precision and gradient scaling are used by the training helpers
when a CUDA device is available.

In [ ]:
from __future__ import annotations

seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import math
import random
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict, Optional, Iterable, Union, Any, Callable, Sequence

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import random_split, Dataset, DataLoader
from torch.amp import autocast, GradScaler

random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using {device} for inference")

torch.set_float32_matmul_precision("high")

#### Plotting and display utilities

Import the plotting components used for reconstruction diagnostics and set the
tabular display limits used during inspection.

In [ ]:
import copy
from dataclasses import dataclass
import time

import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.patches import Polygon, FancyBboxPatch, Rectangle, FancyArrowPatch, Circle
from matplotlib.lines import Line2D


pd.options.display.max_columns = None
pd.options.display.max_rows = 100

### 3.2 Surface representation and forecast windows

The autoencoder works with standardised **log time-scaled IV**,

$$
x_t(m,\tau)=\frac{\log\Xi_t(m,\tau)-\mu_{\log\Xi}}{s_{\log\Xi}},
$$

where the location and scale are estimated from the training data. Unobserved
mesh locations are filled with zero after normalisation and are distinguished
from observations by a separate mask. Decoder coordinates are ordered as
**`(moneyness, maturity)`**, with maturity measured in years.

| Dataset mode | Returned objects |
| :-- | :-- |
| `AE` | One surface grid, its mask, flattened versions, and date |
| `Diff` | Observed history, future targets, latent paths, current return, and origin-date features |

For diffusion, the forecast origin is the close of date $t$: history includes
that date, while targets begin at $t+1$. A window is admitted only when its full
history is available and its final target respects the configured end date.

The surface-state features describe level, ATM level, wing differences,
curvature, and term slope in normalised log-$\Xi$ space, together with their
5- and 21-observation changes. Their scaling statistics use the training-date
cutoff recorded in `feature_stats`.

In [ ]:
class TrajectoryDataset(Dataset):
    """
    One dataset that supports:
      mode="AE":   returns single-day grids + flats
      mode="Diff": returns EOD-consistent history/future sequences
                   (forecast origin = t, after observing close of day t)
                   > history:
                       x_{t-N+1}, ..., x_t
                       r_{t-M+1}, ..., r_t
                       f_{t-L+1}, ..., f_t
                   > future:
                       x_{t+1}, ..., x_{t+H}
                       r_{t+1}, ..., r_{t+H}
                   > OPTIONAL: if precomputed_latents are provided, diff mode
                               returns z_hist and z_fut directly (instead of x)
    Expected input:
      iv_df: long dataframe with columns ["date","ttm","moneyness","t-iv", "log_ret", ...]
    """
    def __init__(
        self,
        iv_df: pd.DataFrame,
        mode: str,
        mesh: Tuple[np.ndarray, np.ndarray],
        stats: Tuple[float, float],
        feature_stats: Optional[Dict[str, dict]] = None,  # (mean, std) for extra features
    ):
        assert mode in ("AE", "Diff")
        self.mode = mode
        self.num_preds = None

        df = iv_df.copy()
        df["date"] = pd.to_datetime(df["date"])
        df["ttm"] = pd.to_numeric(df["ttm"], errors="raise")
        df["moneyness"] = pd.to_numeric(df["moneyness"], errors="raise")
        ttm_vals = np.array(sorted(df["ttm"].unique()), dtype=np.float32)
        m_vals   = np.array(sorted(df["moneyness"].unique()), dtype=np.float32)
        assert (ttm_vals == mesh[0]).all()
        assert (m_vals == mesh[1]).all()
        ttm_to_i = {str(v)[:5]: i for i, v in enumerate(ttm_vals.tolist())}
        m_to_j = {str(v)[:5]: j for j, v in enumerate(m_vals.tolist())}
        m_mesh, ttm_mesh = np.meshgrid(m_vals, ttm_vals, indexing="xy")
        coords = np.stack([m_mesh.reshape(-1), ttm_mesh.reshape(-1)], axis=-1)
        self.coords = torch.tensor(coords, dtype=torch.float32)

        self.dates = []
        X, M = [], []
        R = [] if self.mode == "Diff" else None
        F = [] if self.mode == "Diff" else None
        if self.mode == "Diff":
            base_cols = {"date", "ttm", "moneyness", "t-iv", "log_ret"}
            self.feature_cols = [c for c in df.columns if c not in base_cols]
            self.num_features = len(self.feature_cols)
            assert feature_stats is not None
            self.feature_stats = feature_stats
        # Assemble one masked rectangular surface per date.
        for d, g in df.groupby("date", sort=True):
            grid = np.zeros((len(ttm_vals), len(m_vals)), dtype=np.float32)
            mask = np.zeros((len(ttm_vals), len(m_vals)), dtype=np.float32)
            ii = g["ttm"].astype(np.float32).map(lambda x: str(x)[:5]).map(ttm_to_i).to_numpy(np.int64)
            jj = g["moneyness"].astype(np.float32).map(lambda x: str(x)[:5]).map(m_to_j).to_numpy(np.int64)
            vv = np.log(g["t-iv"].to_numpy(np.float32))
            grid[ii, jj] = vv
            mask[ii, jj] = 1.0
            self.dates.append(pd.to_datetime(d))
            X.append(grid)
            M.append(mask)
            if self.mode == "Diff":
                rvals = g["log_ret"].to_numpy(np.float32)
                if not np.allclose(rvals, rvals[0], atol=0, rtol=0):
                    raise ValueError(f"log_ret not constant within date {d}")
                R.append(float(rvals[0]))
                f_row = []
                for col in self.feature_cols:
                    vals = g[col].to_numpy(np.float32)
                    if not np.all(np.isfinite(vals)):
                        raise ValueError(f"Feature '{col}' contains NaNs or infs at date {d}.")
                    if not np.allclose(vals, vals[0], atol=0, rtol=0):
                        raise ValueError(f"Feature '{col}' not constant within date {d}")
                    f_row.append(float(vals[0]))
                F.append(f_row)

        X = np.stack(X, axis=0)   # (Tdays, T, K)
        M = np.stack(M, axis=0)   # (Tdays, T, K)
        # Normalise observed log-Xi values; the mask identifies zero-filled gaps.
        self.mean, self.std = float(stats[0]), float(stats[1])
        X = np.where(M == 1.0, (X - self.mean) / self.std, 0.0)
        self.X = torch.from_numpy(X).float()  # (Tdays, T, K)
        self.M = torch.from_numpy(M).float()  # (Tdays, T, K)

        if self.mode == "Diff":
            R = np.asarray(R, dtype=np.float32)
            r_spec = self.feature_stats.get("log_ret", {})
            R = TrajectoryDataset._apply_transform(R, r_spec, "log_ret")
            r_mean = float(r_spec["mean"])
            r_std = max(float(r_spec["std"]), 1e-8)
            R = (R - r_mean) / r_std
            self.return_mean = r_mean
            self.return_std = r_std
            self.returns = torch.from_numpy(R).float()

            self.F = np.asarray(F, dtype=np.float32)
            self.F_map = {}
            if self.F.shape != (len(self.dates), self.num_features):
                raise ValueError(f"Feature matrix has shape {self.F.shape}, "
                                 f"expected ({len(self.dates)}, {self.num_features})")
            for j, col in enumerate(self.feature_cols):
                self.F_map[col] = j
                spec = self.feature_stats.get(col, {})
                self.F[:, j] = TrajectoryDataset._apply_transform(self.F[:, j], spec, col)
                if not spec.get("binary", False):
                    mu = float(spec["mean"])
                    sig = max(float(spec["std"]), 1e-8)
                    self.F[:, j] = (self.F[:, j] - mu) / sig

            # Append surface descriptors and scale them using the training-date cutoff.
            surface_state_raw, surface_state_names = TrajectoryDataset.make_surface_state_features(
                        X=X,M=M,ttm_vals=ttm_vals,m_vals=m_vals)
            surface_state_norm = np.empty_like(surface_state_raw, dtype=np.float32)
            dates_arr = pd.to_datetime(pd.Index(self.dates))
            train_end_date = pd.Timestamp(self.feature_stats["__train_end_date__"])
            train_mask = dates_arr <= train_end_date
            old_dim = self.F.shape[1]
            for j, col in enumerate(surface_state_names):
                self.F_map[col] = old_dim + j
                self.feature_cols.append(col)
                self.num_features += 1
                x_train = surface_state_raw[train_mask, j]
                mu = float(np.mean(x_train))
                sd = float(np.std(x_train))
                sd = max(sd, 1e-8)
                if col not in self.feature_stats:
                    self.feature_stats[col] = {"mean": mu,"std": sd,"transform": None,"binary": False}
                else:
                    assert self.feature_stats[col]["mean"] == mu
                    assert self.feature_stats[col]["std"] == sd
                surface_state_norm[:, j] = (surface_state_raw[:, j] - mu) / sd
                if not np.all(np.isfinite(surface_state_norm)):
                    bad = np.where(~np.isfinite(surface_state_norm))
                    raise ValueError(f"Non-finite normalised surface-state features at {bad}.")

            self.F = np.concatenate([self.F.astype(np.float32), surface_state_norm.astype(np.float32)],axis=1)
            assert self.num_features == self.F.shape[1] == len(self.feature_cols)

    @staticmethod
    def make_surface_state_features(X: np.ndarray, M: np.ndarray, ttm_vals: np.ndarray, m_vals: np.ndarray):
        X = np.asarray(X, dtype=np.float32)
        M = np.asarray(M, dtype=np.float32)
        ttm_vals = np.asarray(ttm_vals, dtype=np.float32)
        m_vals = np.asarray(m_vals, dtype=np.float32)

        atm_j = int(np.argmin(np.abs(m_vals - 0.0)))
        left_cols = np.where((m_vals >= -1.0) & (m_vals <= -0.5))[0]
        right_cols = np.where((m_vals >= 0.1) & (m_vals <= 0.5))[0]
        short_rows = np.where(ttm_vals <= (3.0 / 12.0 + 1e-6))[0]
        long_rows = np.where(ttm_vals >= (9.0 / 12.0 - 1e-6))[0]

        if len(left_cols) == 0:
            raise ValueError("No left-wing columns found.")
        if len(right_cols) == 0:
            raise ValueError("No right-wing columns found.")
        if len(short_rows) == 0:
            raise ValueError("No short-maturity rows found.")
        if len(long_rows) == 0:
            raise ValueError("No long-maturity rows found.")

        def masked_mean(arr, mask, axis=None):
            num = np.sum(arr * mask, axis=axis)
            den = np.sum(mask, axis=axis)
            return num / np.maximum(den, 1e-8)

        avg_level = masked_mean(X, M, axis=(1, 2))
        atm_level = masked_mean(X[:, :, atm_j],M[:, :, atm_j],axis=1)
        left_level = masked_mean(X[:, :, left_cols],M[:, :, left_cols],axis=(1, 2))
        right_level = masked_mean(X[:, :, right_cols],M[:, :, right_cols],axis=(1, 2))
        skew_lr = left_level - right_level
        curvature = 0.5 * (left_level + right_level) - atm_level
        atm_short = masked_mean(X[:, short_rows, atm_j],M[:, short_rows, atm_j],axis=1)
        atm_long = masked_mean(X[:, long_rows, atm_j],M[:, long_rows, atm_j],axis=1)
        term_slope_atm = atm_long - atm_short

        D = np.stack([avg_level,atm_level,skew_lr,curvature,term_slope_atm],axis=1).astype(np.float32)
        base_names = ["avg_level_norm_log_tiv","atm_level_norm_log_tiv","skew_left_minus_right_norm_log_tiv",
                      "curvature_wings_minus_atm_norm_log_tiv","term_slope_long_minus_short_atm_norm_log_tiv"]
        if not np.all(np.isfinite(D)):
            bad = np.where(~np.isfinite(D))
            raise ValueError(f"Non-finite base surface descriptors at indices {bad}.")

        features = [D]
        names = list(base_names)

        for lag in (5, 21):
            delta = np.zeros_like(D, dtype=np.float32)
            delta[lag:] = D[lag:] - D[:-lag]
            features.append(delta)
            names.extend([f"d{lag}_{name}" for name in base_names])
            move_norm = np.sqrt(np.mean(delta ** 2, axis=1, keepdims=True)).astype(np.float32)
            features.append(move_norm)
            names.append(f"descriptor_move_norm_{lag}d_norm_log_tiv")
        F = np.concatenate(features, axis=1).astype(np.float32)

        if not np.all(np.isfinite(F)):
            bad = np.where(~np.isfinite(F))
            raise ValueError(f"Non-finite surface state features at indices {bad}.")

        return F, names

    @staticmethod
    def _apply_transform(x: np.ndarray, spec: dict, colname: str) -> np.ndarray:
        transform = spec.get("transform", None)
        if transform is None:
            return x
        if transform == "log":
            if np.any(x <= 0):
                raise ValueError(f"Column '{colname}' has non-positive values but transform='log'.")
            return np.log(x)
        if transform == "log1p":
            if np.any(x <= -1):
                raise ValueError(f"Column '{colname}' has values <= -1 but transform='log1p'.")
            return np.log1p(x)
        raise ValueError(f"Unknown transform '{transform}' for column '{colname}'.")

    def update_diff_params(self, num_surf_hist:int,
                           feature_cols:List[str],
                           num_preds:int,
                           origin_start_date:str,
                           target_end_date:str,
                           precomputed_latents:np.ndarray,
                           target_mode:str):
        assert isinstance(num_surf_hist, int) and num_surf_hist >= 2
        assert isinstance(num_preds, int) and num_preds > 0
        assert target_mode in ("adz", "dz", "z")
        self.num_surf_hist = num_surf_hist
        self.num_preds     = num_preds
        self.target_mode   = target_mode

        for ft in feature_cols:
            assert ft in self.F_map.keys()
        if len(feature_cols) > 0:
            F = self.F[:, [self.F_map[ft] for ft in feature_cols]]
            self.features = torch.from_numpy(F).float()
        else:
            self.features = None

        # Latents must follow exactly the same daily chronology as the surface data.
        Z = np.asarray(precomputed_latents, dtype=np.float32)
        if Z.shape[0] != len(self.dates):
            raise ValueError(f"precomputed_latents has {Z.shape[0]} dates, "
                             f"but dataset has {len(self.dates)} dates.")
        self.Z = torch.from_numpy(Z).float()

        # Admit an origin only when its complete history and future horizon fit.
        self.t0 = max(self.num_surf_hist, 1) - 1
        max_t = len(self.dates) - self.num_preds - 1
        if self.t0 > max_t:
            raise ValueError("No valid forecast origin remains after accounting for history and horizon.")
        origin_start_date = pd.Timestamp(origin_start_date)
        target_end_date = pd.Timestamp(target_end_date)
        valid_t_indices = []
        for t in range(self.t0, max_t + 1):
            origin_date = self.dates[t]
            last_pred_date = self.dates[t + self.num_preds]

            if origin_date < origin_start_date:
                continue
            if last_pred_date > target_end_date:
                continue
            valid_t_indices.append(t)
        if len(valid_t_indices) == 0:
            raise ValueError("No valid forecast origins after applying date filters.")
        self.valid_t_indices = valid_t_indices

    def __len__(self) -> int:
        T = self.X.shape[0]
        if self.mode == "AE":
            return T
        if self.mode == "Diff":
            if self.num_preds is None:
                return 0
            return len(self.valid_t_indices)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        if self.mode == "AE":
            x = self.X[idx].unsqueeze(0) # (1,T,K)
            m = self.M[idx].unsqueeze(0) # (1,T,K)
            out = {
                "x_grid": x,                 # (1,T,K)
                "m_grid": m,                 # (1,T,K)
                "x_flat": x.reshape(-1),      # (P,)
                "m_flat": m.reshape(-1), # (P,)
                "date": str(self.dates[idx].date()),
            }
            return out

        if self.mode == "Diff":
            t = self.valid_t_indices[idx]
            hist_start = t - self.num_surf_hist + 1
            hist_end = t + 1
            fut_start = t + 1
            fut_end = t + 1 + self.num_preds
            ret_t = self.returns[t].unsqueeze(-1)
            ret_fut = self.returns[fut_start : fut_end].unsqueeze(-1)

            out = {
                "ret_t": ret_t,
                "ret_fut": ret_fut,                                                           # (H,1)
                "t_idx": torch.tensor(t, dtype=torch.long),
                "date_origin": str(self.dates[t].date()),
                "date_first_pred": str(self.dates[fut_start].date()),
                "date_last_pred": str(self.dates[fut_end - 1].date())
            }
            if True: # for debugging
                out["x_hist_grid"] = self.X[hist_start : hist_end].unsqueeze(1)  # (N,1,T,K)
                out["m_hist_grid"] = self.M[hist_start : hist_end].unsqueeze(1)  # (N,1,T,K)
                out["x_fut_grid"] = self.X[fut_start : fut_end].unsqueeze(1)      # (H,1,T,K)
                out["m_fut_grid"] = self.M[fut_start : fut_end].unsqueeze(1)      # (H,1,T,K)

            if True: # for debugging
                z_hist = self.Z[hist_start:hist_end]      # (N,d)
                z_fut = self.Z[fut_start:fut_end]         # (H,d)

                # ADZ is anchored at the first history state, not the forecast-origin state.
                z_first = z_hist[0]
                z_last = z_hist[-1]
                anchored_dz_hist = z_hist - z_first.unsqueeze(0)
                dz_hist = torch.cat([torch.zeros_like(z_hist[:1]), z_hist[1:] - z_hist[:-1]], dim=0)
                anchored_dz_fut = z_fut - z_first.unsqueeze(0) # (H,d)
                dz_fut = torch.cat([z_fut[:1] - z_last.unsqueeze(0),  # (1, d)
                                    z_fut[1:] - z_fut[:-1]], dim=0)   # and (H-1, d)

                out["z_first"] = z_first
                out["z_hist"] = z_hist
                out["anchored_dz_hist"] = anchored_dz_hist
                out["dz_hist"] = dz_hist
                out["z_last"] = z_last
                out["z_fut"] = z_fut
                out["anchored_dz_fut"] = anchored_dz_fut
                out["dz_fut"] = dz_fut

                if self.target_mode == "adz":
                    target_z_part = anchored_dz_fut
                elif self.target_mode == "z":
                    target_z_part = z_fut
                elif self.target_mode == "dz":
                    target_z_part = dz_fut
                # Joint target: latent channels first, then the standardised daily return.
                out["target"] = torch.cat([target_z_part, ret_fut], dim=-1)

            if self.features is not None:
                out["feat_static"] = self.features[t]

            return out

#### Align surfaces and return features

Join the processed tables by date and drop rows without complete feature data.
The existing assertion checks the expected initial feature-window loss. The
sorted maturity and moneyness arrays define a common mesh for all datasets.

In [ ]:
# Load data
df_SPX_tIVS = pd.read_csv(f"{PATH_SPX_tIVS}.csv")
df_SPX_tIVS["date"] = pd.to_datetime(df_SPX_tIVS["date"], format="%Y-%m-%d")
df_SPX_tIVS = df_SPX_tIVS.sort_values("date").reset_index(drop=True)
df_SPX_ret = pd.read_csv(f"{PATH_SPX_RETURNS}.csv")
df_SPX_ret["date"] = pd.to_datetime(df_SPX_ret["date"], format="%Y-%m-%d")
df_SPX_full = df_SPX_tIVS.merge(df_SPX_ret, on="date", how="left")
old_len = len(df_SPX_full)
df_SPX_full = df_SPX_full.dropna().reset_index(drop=True)
assert df_SPX_full.isna().sum().sum()==0
assert old_len-len(df_SPX_full) == 170*25
df_SPX_mesh = (np.array(sorted(df_SPX_full["ttm"].unique()), dtype=np.float32),
               np.array(sorted(df_SPX_full["moneyness"].unique()), dtype=np.float32))

#### Training statistics and dataset instances

Surface normalisation and return-feature statistics use data through the end
of **2022**. Non-binary features are scaled after their specified transforms:
selected volatility features use logs and squared returns use `log1p`.
Daily feature rows are deduplicated before these statistics are computed.

The surface datasets use dates through 2022 for training and dates from 2023
for testing. The diffusion dataset instances initially retain the full
chronology; forecast-origin filters are applied later, after latent encoding.
The 2022 validation slice overlaps the training period. The configured final
model fits pass no validation loader, so this slice is not an independent
holdout used by those fits.

In [ ]:
# IVS
df_train = df_SPX_full[df_SPX_full["date"].dt.year <= 2022].reset_index(drop=True)
df_train_log_stats = (float(np.log(df_train["t-iv"].astype(np.float32)).mean()),
                      float(np.log(df_train["t-iv"] .astype(np.float32)).std() + 1e-8))
df_val = df_SPX_full[df_SPX_full["date"].dt.year == 2022].reset_index(drop=True)
df_test = df_SPX_full[df_SPX_full["date"].dt.year >= 2023].reset_index(drop=True)
IVS_TRAIN = TrajectoryDataset(df_train, "AE", mesh=df_SPX_mesh, stats=df_train_log_stats)
IVS_VAL = TrajectoryDataset(df_val, "AE", mesh=df_SPX_mesh, stats=df_train_log_stats)
IVS_TEST = TrajectoryDataset(df_test, "AE", mesh=df_SPX_mesh, stats=df_train_log_stats)


# TRAJ
feature_cols = ["ret_neg_ind", "ret_sq", "cumret_5d", "cumret_21d",
                "rv_5d", "rv_21d", "dsv_21d", "ret_skew_21d", "vov_21d_on_rv5", "min_21d", "max_21d"]
log_transform_cols = {"rv_5d", "rv_21d", "dsv_21d", "vov_21d_on_rv5"}
log1p_transform_cols = {"ret_sq"}
binary_cols = {"ret_neg_ind"}
# Estimate feature statistics once per date
train_daily_feats = (df_train[["date", "log_ret"] + feature_cols]
                     .drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True))
feature_stats = {}
for col in ["log_ret"] + feature_cols:
    x = train_daily_feats[col].to_numpy(np.float32)
    if col in log_transform_cols:
        transform = "log"
    elif col in log1p_transform_cols:
        transform = "log1p"
    else:
        transform = None
    binary = col in binary_cols
    if transform == "log":
        if np.any(x <= 0): raise ValueError(f"Column '{col}' has non-positive values but transform='log'.")
        x_for_stats = np.log(x)
    elif transform == "log1p":
        if np.any(x <= -1): raise ValueError(f"Column '{col}' has values <= -1 but transform='log1p'.")
        x_for_stats = np.log1p(x)
    else:
        x_for_stats = x
    mu = float(np.nanmean(x_for_stats))
    sd = float(np.nanstd(x_for_stats) + 1e-8)
    sd = 1.0 if sd < 1e-8 else sd
    feature_stats[col] = {"binary": binary, "transform": transform, "mean": mu, "std": sd}
feature_stats["__train_end_date__"] = "2022-12-31"

# Keep the full chronology here; diffusion origin filters are applied later.
DIFF_IVS = TrajectoryDataset(df_SPX_full, "AE", mesh=df_SPX_mesh, stats=df_train_log_stats)
DIFF_TRAIN = TrajectoryDataset(df_SPX_full, "Diff", mesh=df_SPX_mesh, stats=df_train_log_stats,
                               feature_stats=feature_stats)
DIFF_VAL = TrajectoryDataset(df_SPX_full, "Diff", mesh=df_SPX_mesh, stats=df_train_log_stats,
                               feature_stats=feature_stats)
DIFF_TEST = TrajectoryDataset(df_SPX_full, "Diff", mesh=df_SPX_mesh, stats=df_train_log_stats,
                               feature_stats=feature_stats)

<a id="autoencoder"></a>
## 4. Surface autoencoder

The autoencoder separates surface geometry from temporal forecasting. A residual
convolutional encoder compresses each masked grid to an eight-dimensional latent
state. A coordinate-based decoder reconstructs the surface and supports the
derivatives needed by the static-arbitrage penalties.

### 4.1 Surface batches

Training batches are shuffled and omit the final incomplete batch. Validation
and test loaders preserve chronological order and retain all observations.

In [ ]:
IVS_TRAIN_LOADER = DataLoader(IVS_TRAIN, batch_size=64, shuffle=True,
                              drop_last=True, num_workers=4, pin_memory=True)
IVS_VAL_LOADER = DataLoader(IVS_VAL, batch_size=64, shuffle=False,
                            drop_last=False, num_workers=4, pin_memory=True)
IVS_TEST_LOADER = DataLoader(IVS_TEST, batch_size=64, shuffle=False,
                             drop_last=False, num_workers=4, pin_memory=True)

### 4.2 Encoder and continuous decoder

The encoder receives the normalised surface and observation mask as separate
channels. The decoder receives a latent state and query coordinates, so its
output can be evaluated on both the reconstruction grid and a denser diagnostic
grid.


#### Residual convolutional encoder

`ResBlock2D` supplies the residual building block. `SurfaceEncoderCNN` stacks
these blocks and maps the result to a latent vector. The selected configuration
uses average pooling and the deterministic `RAE` head; the class also retains
optional coordinate channels, a flattening head, and a variational head.

In [ ]:
class ResBlock2D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, downsample: bool = False, groups: int = 8):
        super().__init__()
        stride = 2 if downsample else 1

        self.gn1 = nn.GroupNorm(num_groups=min(groups, in_ch), num_channels=in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)

        self.gn2 = nn.GroupNorm(num_groups=min(groups, out_ch), num_channels=out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1)

        if (in_ch != out_ch) or downsample:
            self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0)
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.gn1(x)))
        h = self.conv2(F.silu(self.gn2(h)))
        return h + self.skip(x)


class SurfaceEncoderCNN(nn.Module):
    def __init__(
        self,
        latent_dim: int = 8,
        base_channels: int = 32,
        groups: int = 8,
        head_layernorm: bool = False,
        head_dropout: float = 0.0,
        arch: str = "RAE",            # "RAE" or "VAE"
        add_coord_channels: bool = False,
        ttm_vals: Optional[np.ndarray] = None,
        m_vals: Optional[np.ndarray] = None,
        coord_norm: str = "zscore",   # "zscore" or "minmax"
        head_type: str = "avg",       # "avg" or "flatten"
        shallow: bool = False
    ):

        super().__init__()
        self.arch = arch
        self.add_coord_channels = add_coord_channels
        self.head_type = head_type
        self.shallow = shallow
        c = base_channels

        in_channels = 2
        if add_coord_channels:
            if ttm_vals is None or m_vals is None:
                raise ValueError("Need ttm_vals and m_vals when add_coord_channels=True")

            m_mesh, ttm_mesh = np.meshgrid(m_vals, ttm_vals, indexing="xy")  # both (T, K)

            if coord_norm == "zscore":
                m_ch = (m_mesh - m_mesh.mean()) / (m_mesh.std() + 1e-8)
                ttm_ch = (ttm_mesh - ttm_mesh.mean()) / (ttm_mesh.std() + 1e-8)
            elif coord_norm == "minmax":
                m_ch = 2.0 * (m_mesh - m_mesh.min()) / (m_mesh.max() - m_mesh.min() + 1e-8) - 1.0
                ttm_ch = 2.0 * (ttm_mesh - ttm_mesh.min()) / (ttm_mesh.max() - ttm_mesh.min() + 1e-8) - 1.0
            else:
                raise ValueError("coord_norm must be 'zscore' or 'minmax'")

            coord_grid = np.stack([m_ch, ttm_ch], axis=0)  # (2,T,K)
            self.register_buffer("coord_grid", torch.from_numpy(coord_grid).float(), persistent=False)
            in_channels += 2

        self.stem = nn.Conv2d(in_channels, c, kernel_size=3, stride=1, padding=1)
        self.b1_1 = ResBlock2D(c, c, downsample=False, groups=groups)
        self.b1_2 = ResBlock2D(c, c, downsample=False, groups=groups)
        self.b2_1 = ResBlock2D(c, 2*c, downsample=True, groups=groups)
        self.b2_2 = ResBlock2D(2*c, 2*c, downsample=False, groups=groups)
        if not self.shallow:
            self.b3_1 = ResBlock2D(2*c, 4*c, downsample=True, groups=groups)
        else:
            self.b3_1 = ResBlock2D(2*c, 4*c, downsample=False, groups=groups)
        self.b3_2 = ResBlock2D(4*c, 4*c, downsample=False, groups=groups)

        if head_type == "avg":
            self.pool = nn.AdaptiveAvgPool2d(1)
            head_in = 4 * c
        elif head_type == "flatten":
            self.pool = nn.Identity()
            if not self.shallow:
                head_in = 4 * c * 3 * 6
            else:
                head_in = 4 * c * 6 * 11
        else:
            raise ValueError("head_type must be 'avg' or 'flatten'")

        head = []
        if head_layernorm:
            head.append(nn.LayerNorm(head_in))
        head += [
            nn.Linear(head_in, 4*c),
            nn.SiLU(),
        ]
        if head_dropout > 0:
            head.append(nn.Dropout(head_dropout))

        if self.arch=="RAE":
            head.append(nn.Linear(4*c, latent_dim))
        elif self.arch=="VAE":
            head.append(nn.Linear(4*c, 2 * latent_dim))
        else:
            raise ValueError
        self.head = nn.Sequential(*head)

    def forward(self, x_grid: torch.Tensor, m_grid: torch.Tensor) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:

        parts = [x_grid, m_grid]  # each (B,1,T,K)
        if self.add_coord_channels:
            B = x_grid.shape[0]
            coord = self.coord_grid.unsqueeze(0).expand(B, -1, -1, -1)  # (B,2,T,K)
            parts.append(coord)
        x = torch.cat(parts, dim=1)

        h = self.stem(x)
        h = self.b1_2(self.b1_1(h))
        h = self.b2_2(self.b2_1(h))
        h = self.b3_2(self.b3_1(h))

        h = self.pool(h)
        h = h.flatten(1)
        if self.arch=="RAE":
            z = self.head(h)  # (B, latent_dim)
            return z
        elif self.arch=="VAE":
            params = self.head(h)  # (B,2d)
            mu, logvar = params.chunk(2, dim=-1)
            return mu, logvar
        else:
            raise ValueError

#### Fourier features and latent modulation

The decoder embeds the query coordinates with fixed random Fourier features.
At each hidden layer, latent-dependent scale and shift parameters modulate the
coordinate representation (FiLM). Its scalar output is normalised log-$\Xi$.
Smooth coordinate dependence allows the derivative-based loss to use autograd.


In [ ]:
# --------------------------
# Decoder
# --------------------------
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int = 2, num_frequencies: int = 16, scale: float = 3.0):
        super().__init__()
        B = torch.randn(in_dim, num_frequencies) * scale
        self.register_buffer("B", B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., 2)
        proj = 2.0 * math.pi * x @ self.B  # (..., F)
        return torch.cat([x, torch.sin(proj), torch.cos(proj)], dim=-1)


class FiLMNeuralFieldDecoder(nn.Module):
    def __init__(
        self,
        latent_dim: int = 8,
        hidden_dim: int = 256,
        num_layers: int = 5,
        num_frequencies: int = 16,
        fourier_scale: float = 3.0,
        coord_mean: tuple[float, float] = (None, None),
        coord_std:  tuple[float, float] = (None, None)
    ):
        super().__init__()
        self.ff = FourierFeatures(in_dim=2, num_frequencies=num_frequencies, scale=fourier_scale)
        in_dim = 2 + 2 * num_frequencies

        self.register_buffer("coord_mean", torch.tensor(coord_mean, dtype=torch.float32))
        self.register_buffer("coord_std",  torch.tensor(coord_std,  dtype=torch.float32))

        self.layers = nn.ModuleList()
        self.films  = nn.ModuleList()

        self.layers.append(nn.Linear(in_dim, hidden_dim))
        self.films.append(nn.Linear(latent_dim, 2 * hidden_dim))

        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.films.append(nn.Linear(latent_dim, 2 * hidden_dim))

        # Zero modulation weights start each layer close to identity FiLM modulation.
        for film in self.films:
            nn.init.zeros_(film.weight)
            nn.init.zeros_(film.bias)

        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, coords: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        if coords.dim() == 2:
            coords = coords.unsqueeze(0).expand(z.size(0), -1, -1)  # (B,P,2)

        coords = (coords - self.coord_mean) / (self.coord_std + 1e-8)

        x = self.ff(coords)  # (B,P,D)
        h = x

        for lin, film in zip(self.layers, self.films):
            h = lin(h)  # (B,P,H)
            gamma_beta = film(z)  # (B,2H)
            gamma, beta = gamma_beta.chunk(2, dim=-1)
            gamma = 1.0 + gamma  # identity-ish init
            h = h * gamma.unsqueeze(1) + beta.unsqueeze(1)
            h = F.silu(h)

        y = self.out(h)           # (B,P,1)
        return y

#### Autoencoder interface

`SurfaceAE` combines the encoder and decoder and exposes encoding, decoding,
and reconstruction methods. It retains both deterministic and variational
branches. The instantiated model below uses `enc_arch="RAE"`; the VAE
reparameterisation path is therefore not used by that fit.


In [ ]:
# --------------------------
# Autoencoder wrapper
# --------------------------
class SurfaceAE(nn.Module):
    def __init__(
        self,
        latent_dim: int = 8,
        enc_base_channels: int = 32,
        enc_groups: int = 8,
        enc_head_layernorm: bool = False,
        enc_head_dropout: float = 0.0,
        enc_arch: str = "RAE",
        enc_add_coord_channels: bool = False,
        enc_coord_norm: str = "zscore",
        enc_head_type: str = "avg",
        enc_shallow: bool = False,
        dec_hidden_dim: int = 256,
        dec_num_layers: int = 5,
        dec_num_freqs: int = 16,
        dec_fourier_scale: float = 3.0,
        ttm_vals: Optional[np.ndarray] = None,
        m_vals: Optional[np.ndarray] = None,
    ):
        super().__init__()
        self.encoder = SurfaceEncoderCNN(
            latent_dim=latent_dim,
            base_channels=enc_base_channels,
            groups=enc_groups,
            head_layernorm=enc_head_layernorm,
            head_dropout=enc_head_dropout,
            arch=enc_arch,
            add_coord_channels=enc_add_coord_channels,
            ttm_vals=ttm_vals,
            m_vals=m_vals,
            coord_norm=enc_coord_norm,
            head_type=enc_head_type,
            shallow=enc_shallow
        )
        self.decoder = FiLMNeuralFieldDecoder(
            latent_dim=latent_dim,
            hidden_dim=dec_hidden_dim,
            num_layers=dec_num_layers,
            num_frequencies=dec_num_freqs,
            fourier_scale=dec_fourier_scale,
            coord_mean=(float(m_vals.mean()), float(ttm_vals.mean())),
            coord_std=(float(m_vals.std() + 1e-8), float(ttm_vals.std() + 1e-8))
        )
        self.arch = enc_arch

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        eps = torch.randn_like(mu)
        std = torch.exp(0.5 * logvar)
        return mu + std * eps

    def encode(self, x_grid: torch.Tensor, m_grid: torch.Tensor,
               deterministic: bool=True, return_stats: bool=False):
        if self.arch == "RAE":
            z = self.encoder(x_grid, m_grid)  # (B,d)
            return (z, None, None) if return_stats else z
        elif self.arch == "VAE":
            mu, logvar = self.encoder(x_grid, m_grid)  # (B,d), (B,d)
            z = mu if deterministic else self.reparameterize(mu, logvar)
            return (z, mu, logvar) if return_stats else z
        else:
            raise ValueError

    def decode(self, coords: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        # returns (B,P)
        return self.decoder(coords, z).squeeze(-1)

    def forward(self, x_grid: torch.Tensor, m_grid: torch.Tensor, coords: torch.Tensor):
        if self.arch=="RAE":
            z = self.encoder(x_grid, m_grid)  # (B,d)
            y_hat = self.decode(coords, z) # (B,P)
            return z, y_hat
        elif self.arch=="VAE":
            mu, logvar = self.encoder(x_grid, m_grid) # (B,d), (B,d)
            z = self.reparameterize(mu, logvar) # (B,d)
            y_hat = self.decode(coords, z)  # (B,P)
            return mu, logvar, z, y_hat
        else:
            raise ValueError

    @torch.no_grad()
    def forward_deterministic(self, x_grid: torch.Tensor, m_grid: torch.Tensor, coords: torch.Tensor):
        if self.arch=="RAE":
            z = self.encoder(x_grid, m_grid) # (B,d)
            y_hat = self.decode(coords, z) # (B,P)
            return z, y_hat
        elif self.arch=="VAE":
            mu, logvar = self.encoder(x_grid, m_grid)  # (B,d), (B,d)
            y_hat = self.decode(coords, mu) # (B,P)
            return mu, logvar, mu, y_hat
        else:
            raise ValueError

### 4.3 Reconstruction objective and arbitrage penalties

Reconstruction error is measured only at observed lattice points. Static-arbitrage
penalties are evaluated separately on a denser set of coordinates, allowing the
decoder's shape between observations to contribute to training.


#### Dense coordinates for shape checks

Densify moneyness within each observed maturity slice, then add intermediate
maturities at matching moneyness values. These coordinates are used for
derivative penalties, while reconstruction loss continues to use the original
observed grid. A plotting helper displays the two coordinate sets.


In [ ]:
# ----------------------------
# No-Arb Helpers
# ----------------------------
def densify_x_per_tau_slice(mesh, mask2d, dx=0.02, include_observed=True):
    ttm_vals, m_vals = mesh
    ttm_vals = np.asarray(ttm_vals, dtype=np.float32)  # (T,)
    m_vals   = np.asarray(m_vals, dtype=np.float32) # (K,)
    mask2d   = np.asarray(mask2d, dtype=np.float32) # (T,K)

    T, K = mask2d.shape
    coords_list = []
    counts = []

    for i, tau in enumerate(ttm_vals):
        obs_j = np.where(mask2d[i, :] > 0.5)[0]
        if obs_j.size == 0:
            counts.append(0)
            continue

        xs_obs = m_vals[obs_j]
        xs_out = []

        if include_observed:
            xs_out.extend(xs_obs.tolist())

        for a, b in zip(xs_obs[:-1], xs_obs[1:]):
            interior = np.arange(a + dx, b - 1e-12, dx, dtype=np.float32)  # strictly inside
            if interior.size:
                xs_out.extend(interior.tolist())

        xs_out = np.unique(np.asarray(xs_out, dtype=np.float32))
        xs_out.sort()

        taus = np.full_like(xs_out, tau, dtype=np.float32)
        coords_list.append(np.stack([xs_out, taus], axis=-1))
        counts.append(len(xs_out))

    coords = np.concatenate(coords_list, axis=0) if coords_list else np.zeros((0,2), np.float32)
    meta = {"counts_per_tau": counts, "total": int(coords.shape[0])}
    return torch.tensor(coords, dtype=torch.float32), meta


def densify_tau_from_coords(coords_in: torch.Tensor, dt=0.02, include_existing=True, x_round=6):
    cd = coords_in.detach().cpu().numpy().astype(np.float32)
    x = cd[:, 0]
    t = cd[:, 1]

    x_key = np.round(x, x_round)
    uniq_x = np.unique(x_key)

    out = []
    counts = []

    for xk in uniq_x:
        sel = (x_key == xk)
        taus = np.unique(t[sel])
        taus.sort()
        if taus.size == 0:
            counts.append(0)
            continue

        taus_out = []
        if include_existing:
            taus_out.extend(taus.tolist())

        for a, b in zip(taus[:-1], taus[1:]):
            interior = np.arange(a + dt, b - 1e-12, dt, dtype=np.float32)  # strictly inside
            if interior.size:
                taus_out.extend(interior.tolist())

        taus_out = np.unique(np.asarray(taus_out, dtype=np.float32))
        taus_out.sort()

        xs = np.full_like(taus_out, xk, dtype=np.float32)
        out.append(np.stack([xs, taus_out], axis=-1))
        counts.append(len(taus_out))

    coords_out = np.concatenate(out, axis=0) if out else np.zeros((0,2), np.float32)
    meta = {"counts_per_x": counts, "total": int(coords_out.shape[0])}
    return torch.tensor(coords_out, dtype=torch.float32), meta


def plot_observed_vs_two_coords(mesh, mask2d, coords_a, coords_b,
                                label_a="x-dense", label_b="x+dense-then-tau",
                                title="Observed vs densified coords"):
    ttm_vals, m_vals = mesh
    ttm_vals = np.asarray(ttm_vals, dtype=np.float32)
    m_vals   = np.asarray(m_vals, dtype=np.float32)
    mask2d   = np.asarray(mask2d, dtype=np.float32)

    # observed points from mask on original grid
    ii, jj = np.where(mask2d > 0.5)
    x_obs = m_vals[jj]
    t_obs = ttm_vals[ii]

    # densified sets
    a = coords_a.detach().cpu().numpy()
    b = coords_b.detach().cpu().numpy()

    plt.figure(figsize=(8,5))
    plt.scatter(a[:,0], a[:,1], s=6,  alpha=0.20, label=label_a)
    plt.scatter(b[:,0], b[:,1], s=6,  alpha=0.20, label=label_b)
    plt.scatter(x_obs, t_obs,   s=30, alpha=0.95, label="observed points")

    plt.xlabel("x (log-moneyness)")
    plt.ylabel("tau (years)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

#### Loss components

For the selected deterministic autoencoder, the training objective combines
masked reconstruction error, latent regularisation, and the weighted
butterfly/calendar penalties:

$$
\mathcal L=\mathcal L_{\mathrm{rec}}
+\lambda_z\,\operatorname{mean}(z^2)
+\lambda_{\mathrm{mom}}\mathcal L_{\mathrm{mom}}
+\lambda_{\mathrm{but}}\mathcal L_{\mathrm{but}}
+\lambda_{\mathrm{cal}}\mathcal L_{\mathrm{cal}}.
$$

`roper_losses_from_logxi` first reverses surface normalisation, then differentiates
the decoder with respect to moneyness and maturity. Negative butterfly quantities
and negative maturity derivatives of log-$\Xi$ receive squared hinge penalties.
These are soft training penalties evaluated at the selected coordinates. The KL
helper belongs to the retained variational alternative.


In [ ]:
# ----------------------------
# Losses
# ----------------------------
def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    num = ((pred - target) ** 2 * mask).sum()
    den = mask.sum().clamp_min(1.0)
    return num / den

def latent_moment_penalty(z: torch.Tensor) -> torch.Tensor:
    mu = z.mean(dim=0)
    var = z.var(dim=0, unbiased=False)
    return (mu**2).mean() + ((var - 1.0)**2).mean()

def kl_standard_normal(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    return 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).mean()


def roper_losses_from_logxi(
    decoder,
    coords_eval,
    z,
    mean: float,
    std: float
):
    device = z.device

    coords = coords_eval.to(device) # (P_dense,2)
    coords = coords.unsqueeze(0).expand(z.size(0), -1, -1).clone()
    coords.requires_grad_(True)

    with autocast("cuda", enabled=False):
        if coords.dtype!=torch.float32: coords = coords.float()
        if z.dtype!=torch.float32: z = z.float()
        # decoder outputs y_hat_norm (normalized logXi)
        y_hat_norm = decoder(coords, z).squeeze(-1)
        y = y_hat_norm * std + mean
        # first derivatives of y
        dy_dcoords = torch.autograd.grad(
            outputs=y.sum(),
            inputs=coords,
            create_graph=True,
            retain_graph=True,
        )[0]
        y_x   = dy_dcoords[..., 0]
        y_tau = dy_dcoords[..., 1]
        # second derivative y_xx
        d2y_dcoords = torch.autograd.grad(
            outputs=y_x.sum(),
            inputs=coords,
            create_graph=True,
            retain_graph=True,
        )[0]
        y_xx = d2y_dcoords[..., 0]
        # Xi and its derivatives via chain rule
        Xi = torch.exp(y)
        Xi_x  = Xi * y_x
        Xi_xx = Xi * (y_xx + y_x.pow(2))
        xcoord = coords[..., 0]
        # Roper IV3: I_Xi >= 0
        I_Xi = (1.0-xcoord*y_x).pow(2) - 0.25*Xi.pow(4)* y_x.pow(2) + Xi*Xi_xx
        loss_but = F.relu(-I_Xi).pow(2).mean()
        # Roper IV4: y_tau >= 0
        loss_cal = F.relu(-y_tau).pow(2).mean()
        # summarize arb violations
        diag = {
            "I_min": float(I_Xi.detach().min().cpu()),
            "I_mean": float(I_Xi.detach().mean().cpu()),
            "y_tau_min": float(y_tau.detach().min().cpu()),
            "y_tau_mean": float(y_tau.detach().mean().cpu()),
        }

    return loss_but, loss_cal, diag

#### One training or validation epoch

Each epoch reports reconstruction and arbitrage-loss components separately.
The training loop supports gradient scaling and clipping. During validation,
ordinary reconstruction is evaluated without gradients, while coordinate
gradients are enabled for the derivative-based shape diagnostics.


In [ ]:
# ----------------------------
# Train / Eval loops
# ----------------------------
def train_one_epoch_ae(model: nn.Module, loader, optimizer, device: torch.device, coords_rec, coords_noarb, *,
                       lam_z_l2: float = 1e-4, lam_z_mom: float = 0.0,
                       no_arb: bool = False, lam_but: float = 0.0, lam_cal: float = 0.0,
                       mean: float = 0.0, std: float = 0.0,
                       grad_clip: float = 1.0, use_amp: bool = True,
                       arch: str="RAE", beta: float = 0.0):
    model.train()
    scaler = train_one_epoch_ae.scaler

    total_loss = total_rec = total_l2 = total_mom = total_kl = total_but = total_cal = 0.0
    n = 0

    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        x_grid = batch["x_grid"].to(device, non_blocking=True)  # (B,1,T,K)
        m_grid = batch["m_grid"].to(device, non_blocking=True)  # (B,1,T,K)
        x_flat = batch["x_flat"].to(device, non_blocking=True)  # (B,P)
        m_flat = batch["m_flat"].to(device, non_blocking=True)  # (B,P)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(use_amp and device.type == "cuda")):
            if arch=="RAE":
                z, y_hat = model(x_grid, m_grid, coords_rec)
                loss_rec = masked_mse(y_hat, x_flat, m_flat)
                loss_l2 = (z**2).mean()
                loss_mom = latent_moment_penalty(z) if lam_z_mom > 0 else torch.tensor(0.0, device=device)
                loss = loss_rec + lam_z_l2 * loss_l2 + lam_z_mom * loss_mom
            elif arch=="VAE":
                mu, logvar, z, y_hat = model(x_grid, m_grid, coords_rec)
                loss_rec = masked_mse(y_hat, x_flat, m_flat)
                loss_kl  = kl_standard_normal(mu, logvar)
                loss = loss_rec + beta * loss_kl
            else:
                raise ValueError

        loss_but, loss_cal, diag = roper_losses_from_logxi(
            decoder=model.decoder,
            coords_eval=coords_noarb,   # (P,2)
            z=z,  # (B,d)
            mean=mean, std=std
        )
        if no_arb:
            loss += lam_but * loss_but + lam_cal * loss_cal

        if device.type == "cuda" and use_amp:
            scaler.scale(loss).backward()
            if grad_clip is not None and grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += float(loss.detach().cpu())
        total_rec += float(loss_rec.detach().cpu())
        total_but += float(loss_but.detach().cpu())
        total_cal += float(loss_cal.detach().cpu())
        n += 1
        loss_dict = {"loss": total_loss/n, "rec": total_rec/n,
                     "but": total_but/n, "cal": total_cal/n,
                     "Imin": diag["I_min"], "dtmin": diag["y_tau_min"]}
        if arch=="RAE":
            total_l2 += float(loss_l2.detach().cpu())
            total_mom += float(loss_mom.detach().cpu()) if lam_z_mom > 0 else 0.0
            loss_dict |= {"l2": total_l2/n, "mom": total_mom/n}
        elif arch=="VAE":
            total_kl += float(loss_kl.detach().cpu())
            loss_dict |= {"kl": total_kl / n, "beta": beta}
        else:
            raise ValueError
        pbar.set_postfix(loss_dict)

    total_dict = {"loss": total_loss/n, "rec": total_rec/n,
                  "but": total_but/n, "cal": total_cal/n}
    if arch=="RAE":
        total_dict |= {"l2": total_l2/n, "mom": total_mom/n}
    elif arch=="VAE":
        total_dict |= {"kl": total_kl / n, "beta": beta}
    else:
        raise ValueError
    return total_dict

def eval_one_epoch_ae(model: nn.Module, loader, device: torch.device,coords_rec, coords_noarb, *,
                     lam_z_l2: float = 1e-4, lam_z_mom: float = 0.0,
                     no_arb: bool = False, lam_but: float = 0.0, lam_cal: float = 0.0,
                     mean: float = 0.0, std: float = 0.0,
                     use_amp: bool = True, arch: str="RAE", beta: float=0.0):
    model.eval()
    total_loss = total_rec = total_l2 = total_mom = total_kl = total_but = total_cal = 0.0
    n = 0

    for batch in tqdm(loader, desc="val", leave=False):
        x_grid = batch["x_grid"].to(device, non_blocking=True)
        m_grid = batch["m_grid"].to(device, non_blocking=True)
        x_flat = batch["x_flat"].to(device, non_blocking=True)
        m_flat = batch["m_flat"].to(device, non_blocking=True)

        with torch.no_grad():
            with autocast("cuda", enabled=(use_amp and device.type == "cuda")):
                if arch=="RAE":
                    z, y_hat = model(x_grid, m_grid, coords_rec)
                    loss_rec = masked_mse(y_hat, x_flat, m_flat)
                    loss_l2 = (z**2).mean()
                    loss_mom = latent_moment_penalty(z) if lam_z_mom > 0 else torch.tensor(0.0, device=device)
                    loss = loss_rec + lam_z_l2 * loss_l2 + lam_z_mom * loss_mom
                elif arch=="VAE":
                    mu, logvar, z, y_hat = model.forward_deterministic(x_grid, m_grid, coords_rec)
                    loss_rec = masked_mse(y_hat, x_flat, m_flat)
                    loss_kl  = kl_standard_normal(mu, logvar)
                    loss = loss_rec + beta * loss_kl
                else:
                    raise ValueError

        with torch.enable_grad():
            loss_but, loss_cal, diag = roper_losses_from_logxi(
                decoder=model.decoder,
                coords_eval=coords_noarb, # (P,2)
                z=z, # (B,d)
                mean=mean, std=std
            )
        if no_arb:
            loss += lam_but * loss_but + lam_cal * loss_cal

        total_loss += float(loss.detach().cpu())
        total_rec += float(loss_rec.detach().cpu())
        total_but += float(loss_but.detach().cpu())
        total_cal += float(loss_cal.detach().cpu())
        if arch == "RAE":
            total_l2  += float(loss_l2.detach().cpu())
            total_mom += float(loss_mom.detach().cpu()) if lam_z_mom > 0 else 0.0
        elif arch == "VAE":
            total_kl  += float(loss_kl.detach().cpu())
        n += 1

    loss_dict = {"loss": total_loss/n, "rec": total_rec/n,
                 "but": total_but/n, "cal": total_cal/n,
                 "Imin": diag["I_min"], "dtmin": diag["y_tau_min"]}
    if arch=="RAE":
        loss_dict |= {"l2": total_l2/n, "mom": total_mom/n}
    elif arch=="VAE":
        loss_dict |= {"kl": total_kl / n, "beta": beta}
    else:
        raise ValueError
    return loss_dict


#### Reconstruction plots

Convert normalised decoder outputs back to time-scaled IV and compare observed,
reconstructed, and residual surfaces. A fixed residual colour range helps
compare selected dates without changing the visual scale between panels.


In [ ]:
"""
Visualization
"""
def compute_fixed_diff_zlim(model, dataset, device, mean, std, idxs, q=0.98, arch="RAE"):
    diffs = []
    for idx in idxs:
        df = recon_to_long_df(model, dataset, idx, device, mean, std, arch)
        x = df["t-iv_diff"].to_numpy()
        x = x[np.isfinite(x)]
        diffs.append(np.abs(x))
    if len(diffs) == 0:
        return (-1.0, 1.0)
    diffs = np.concatenate(diffs, axis=0)
    L = float(np.quantile(diffs, q))
    L = max(L, 1e-6)
    return (-L, L)

def plot_surface_from_df(ax, df: pd.DataFrame, value_col: str, title: str, zlim=None, cmap="viridis", norm=None):
    pivot = df.pivot(index="ttm", columns="moneyness", values=value_col).sort_index().sort_index(axis=1)
    X, Y = np.meshgrid(pivot.columns.to_numpy(dtype=np.float32),
                       pivot.index.to_numpy(dtype=np.float32))
    Z = pivot.to_numpy(dtype=np.float32)
    Zm = np.ma.masked_invalid(Z)
    surf = ax.plot_surface(X, Y, Zm, cmap=cmap, norm=norm, edgecolor="none", linewidth=0, antialiased=True)
    ax.set_title(title)
    ax.set_xlabel("moneyness")
    ax.set_ylabel("ttm (years)")
    ax.set_zlabel("t-iv")
    if zlim is not None:
        ax.set_zlim(*zlim)
    return surf

@torch.no_grad()
def recon_to_long_df(model, dataset, idx: int, device, mean: float, std: float, arch="RAE") -> pd.DataFrame:
    model.eval()
    item = dataset[idx]
    # shapes
    x_grid = item["x_grid"].unsqueeze(0).to(device)  # (1,1,T,K)
    m_grid = item["m_grid"].unsqueeze(0).to(device)  # (1,1,T,K)
    T = x_grid.shape[-2]
    K = x_grid.shape[-1]
    # coords from dataset (single tensor)
    coords = dataset.coords.to(device)  # (P,2)
    P = coords.shape[0]
    if arch=="RAE":
        # forward
        z, y_hat = model(x_grid, m_grid, coords)  # y_hat: (1,P) in normalized log space
    elif arch=="VAE":
        # deterministic recon for stable plots
        mu, logvar, z_det, y_hat = model.forward_deterministic(x_grid, m_grid, coords)  # y_hat: (1,P)
    else:
        raise ValueError
    # true in normalized log space
    x_true = item["x_grid"].to(device).view(1, -1)  # (1,P)
    mask   = item["m_grid"].to(device).view(1, -1)  # (1,P)
    # back to t-iv
    x_tiv = torch.exp(x_true * std + mean).view(-1)  # (P,)
    y_tiv = torch.exp(y_hat  * std + mean).view(-1)  # (P,)
    # apply mask: keep missing as NaN for plotting
    m = mask.view(-1)
    x_tiv = torch.where(m > 0.5, x_tiv, torch.tensor(float("nan"), device=device))
    y_tiv = torch.where(m > 0.5, y_tiv, torch.tensor(float("nan"), device=device))
    d_tiv = y_tiv - x_tiv
    # return
    coords_np = coords.detach().cpu().numpy()
    df = pd.DataFrame({
        "moneyness": coords_np[:, 0],
        "ttm":       coords_np[:, 1],
        "t-iv_true":  x_tiv.detach().cpu().numpy(),
        "t-iv_recon": y_tiv.detach().cpu().numpy(),
        "t-iv_diff":  d_tiv.detach().cpu().numpy(),
    })
    return df

@torch.no_grad()
def plot_true_recon_diff_3d_pivotstyle(model, dataset, device, *,
                                       mean, std, idxs, diff_zlim,
                                       zlim_true=(0.0, 0.6), arch="RAE"):
    """
    For each idx: 1x3 figure: TRUE, RECON, DIFF (recon-true).
    """
    norm_true = mcolors.Normalize(vmin=zlim_true[0], vmax=zlim_true[1])
    norm_diff = mcolors.Normalize(vmin=diff_zlim[0], vmax=diff_zlim[1])

    for idx in idxs:
        df = recon_to_long_df(model, dataset, idx, device, mean, std, arch)

        fig = plt.figure(figsize=(18, 6))
        ax1 = fig.add_subplot(131, projection="3d")
        ax2 = fig.add_subplot(132, projection="3d")
        ax3 = fig.add_subplot(133, projection="3d")

        s1 = plot_surface_from_df(ax1, df, "t-iv_true",  f"TRUE (idx={idx})",
                                  zlim=zlim_true, cmap="viridis", norm=norm_true)
        s2 = plot_surface_from_df(ax2, df, "t-iv_recon", f"RECON (idx={idx})",
                                  zlim=zlim_true, cmap="viridis", norm=norm_true)
        s3 = plot_surface_from_df(ax3, df, "t-iv_diff",  f"DIFF recon-true (idx={idx})",
                                  zlim=diff_zlim, cmap="coolwarm", norm=norm_diff)

        fig.colorbar(s1, ax=ax1, shrink=0.5, aspect=10)
        fig.colorbar(s2, ax=ax2, shrink=0.5, aspect=10)
        fig.colorbar(s3, ax=ax3, shrink=0.5, aspect=10)

        # plt.tight_layout()
        plt.show()

### 4.4 Training and checkpoint reuse

The trainer constructs the dense derivative grid, runs the epoch loops, and
records reconstruction and learning-rate histories. With a validation loader,
checkpoint selection uses validation reconstruction loss; without one, each
epoch overwrites the checkpoint with the current model.


In [ ]:
# ----------------------------
# Main trainer
# ----------------------------
train_one_epoch_ae.scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

def train_surface_ae(model: nn.Module, train_loader, val_loader, *,
                     epochs: int = 100, lr: float = 3e-4, weight_decay: float = 1e-4,
                     lam_z_l2: float = 1e-4, lam_z_mom: float = 0.0,
                     no_arb: bool = False, lam_but: float = 0.0, lam_cal: float = 0.0,
                     grad_clip: float = 1.0, use_amp: bool = True, ckpt_path: str = "surface_ae_best.pt",
                     beta: float=0.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    arch = model.arch

    mean = float(train_loader.dataset.mean)
    std = float(train_loader.dataset.std)

    coords_rec = train_loader.dataset.coords.to(device, non_blocking=True)
    mask2d = train_loader.dataset[0]["m_grid"].squeeze(0).cpu().numpy()
    coords_x_dense, _ = densify_x_per_tau_slice(df_SPX_mesh, mask2d, dx=0.02, include_observed=True)
    coords_xy_dense, _ = densify_tau_from_coords(coords_x_dense, dt=0.02, include_existing=True)
    coords_noarb = coords_xy_dense.to(device, non_blocking=True)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if val_loader is not None:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=8)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.05)

    best_val = float("inf")
    train_rec_hist, val_rec_hist, lr_hist = [], [], []

    if arch=="VAE":
        beta_max = beta
        warmup_epochs = 20

    for ep in range(1, epochs + 1):
        if arch=="VAE":
            beta = beta_max * min(1.0, ep / warmup_epochs)
        tr = train_one_epoch_ae(
            model, train_loader, opt, device, coords_rec, coords_noarb,
            lam_z_l2=lam_z_l2, lam_z_mom=lam_z_mom,
            no_arb=no_arb, lam_but=lam_but, lam_cal=lam_cal,
            mean=mean, std=std,
            grad_clip=grad_clip, use_amp=use_amp,
            arch=arch, beta=beta
        )
        if val_loader is not None:
            va = eval_one_epoch_ae(
                model, val_loader, device, coords_rec, coords_noarb,
                lam_z_l2=lam_z_l2, lam_z_mom=lam_z_mom,
                no_arb=no_arb, lam_but=lam_but, lam_cal=lam_cal,
                mean=mean, std=std,
                use_amp=use_amp, arch=arch, beta=beta
            )
            sched.step(va["rec"])
        else:
            va = None
            sched.step()
        lr_now = opt.param_groups[0]["lr"]
        train_rec_hist.append(tr["rec"])
        if val_loader is not None:
            val_rec_hist.append(va["rec"])
        lr_hist.append(lr_now)

        if arch=="RAE":
            if val_loader is not None:
                print(
                    f"[{ep:03d}] lr={lr_now:.2e} "
                    f"train: loss={tr['loss']:.6f} rec={tr['rec']:.6f} | "
                    f"val: loss={va['loss']:.6f} rec={va['rec']:.6f} | "
                    f"val: but={va['but']:.6f} cal={va['cal']:.6f}"
                )
            else:
                print(
                    f"[{ep:03d}] lr={lr_now:.2e} "
                    f"train: loss={tr['loss']:.6f} rec={tr['rec']:.6f} | "
                    f"train: but={tr['but']:.6f} cal={tr['cal']:.6f}"
                )
        elif arch=="VAE":
            if val_loader is not None:
                print(
                    f"[{ep:03d}] lr={lr_now:.2e} beta={beta:.2e} "
                    f"train: loss={tr['loss']:.6f} rec={tr['rec']:.6f} kl={tr['kl']:.6f} | "
                    f"val: loss={va['loss']:.6f} rec={va['rec']:.6f} kl={va['kl']:.6f} | "
                    f"val: but={va['but']:.6f} cal={va['cal']:.6f}"
                )
            else:
                print(
                    f"[{ep:03d}] lr={lr_now:.2e} beta={beta:.2e} "
                    f"train: loss={tr['loss']:.6f} rec={tr['rec']:.6f} kl={tr['kl']:.6f} | "
                    f"train: but={tr['but']:.6f} cal={tr['cal']:.6f}"
                )
        else:
            raise ValueError

        if val_loader is not None:
            if va["rec"] < best_val:
                best_val = va["rec"]
                torch.save({"model": model.state_dict(), "best_val_rec": best_val, "epoch":ep, "arch":arch}, ckpt_path)
                print(f"  saved best -> {ckpt_path} (val_rec={best_val:.6f})")
        else:
            torch.save({"model": model.state_dict(),"final_train_rec": tr["rec"],"epoch": ep,"arch": arch}, ckpt_path)

        if ep == 1 or ep % 10 == 0:
            if ep == 1:
                fixed_idxs = [0, 10, 25, 50, 100, 150, 200]
                FIXED_DIFF_ZLIM = compute_fixed_diff_zlim(
                    model, val_loader.dataset if val_loader is not None else train_loader.dataset,
                    device, mean=mean, std=std,
                    idxs=fixed_idxs, q=0.98, arch=arch
                )

            plot_true_recon_diff_3d_pivotstyle(
                model, val_loader.dataset if val_loader is not None else train_loader.dataset, device,
                mean=mean, std=std,
                idxs=[0],
                diff_zlim=FIXED_DIFF_ZLIM,
                zlim_true=(0.0, 0.6), arch=arch
            )

    print("done. best val_rec:", best_val)
    plt.figure(figsize=(7,4))
    plt.plot(train_rec_hist, label="train_rec")
    plt.plot(val_rec_hist, label="val_rec")
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.title("Masked recon loss (log scale)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(7,3))
    plt.plot(lr_hist)
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.title("Learning rate")
    plt.tight_layout()
    plt.show()

    return model

#### Configured autoencoder fit

| Setting | Value in this run |
| :-- | :-- |
| Latent dimension / encoder base channels | 8 / 32 |
| Encoder head | Deterministic `RAE`, average pooling |
| Decoder hidden width / layers | 256 / 5 |
| Fourier frequencies / scale | 16 / 3.0 |
| Epochs / learning rate | 200 / $3\times10^{-4}$ |
| Latent $L^2$ weight | $10^{-4}$ |
| Butterfly / calendar weights | $10^{-3}$ / $10^{-4}$ |

Training is skipped when the final checkpoint exists. After either training or
reuse, the checkpoint is loaded, the model is put in evaluation mode, and all
parameters are frozen before the latent series is constructed. This fit passes
`None` as its validation loader.


In [ ]:
AE_MODEL = SurfaceAE(
    latent_dim=8,
    enc_base_channels=32,
    enc_groups=8,
    enc_head_layernorm=False,
    enc_head_dropout=0.0,
    enc_arch="RAE",
    enc_add_coord_channels=False,
    enc_coord_norm="zscore",
    enc_head_type="avg",
    enc_shallow=False,
    dec_hidden_dim=256,
    dec_num_layers=5,
    dec_num_freqs=16,
    dec_fourier_scale=3.0,
    ttm_vals=df_SPX_mesh[0],
    m_vals=df_SPX_mesh[1]
)


if not os.path.exists(PATH_FINAL_AE_MODEL):
    ae_trained = train_surface_ae(
        AE_MODEL,
        IVS_TRAIN_LOADER,
        None,
        epochs=200,
        lr=3e-4,
        weight_decay=1e-4,
        lam_z_l2=1e-4,
        lam_z_mom=0,
        no_arb=True,
        lam_but=1e-3,
        lam_cal=1e-4,
        ckpt_path=PATH_FINAL_AE_MODEL,
        beta=0
    )

ckpt = torch.load(PATH_FINAL_AE_MODEL, map_location=device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
AE_MODEL.load_state_dict(state, strict=True)
AE_MODEL.to(device)
AE_MODEL.eval()
for p in AE_MODEL.parameters():
    p.requires_grad_(False)

<a id="diffusion"></a>
## 5. Latent trajectory diffusion

The second model learns the joint conditional distribution of future surface
states and index returns. It denoises an entire future path at once, with the
observed latent history and origin-date state providing the conditioning
information. Surface reconstruction and trajectory generation are trained in
separate stages.


### 5.1 Latent data and target convention

Let $z_t\in\mathbb{R}^8$ denote an autoencoder state, $L=21$ the history length,
and $H=30$ the forecast horizon. The selected `adz` target is anchored to the
**first history state**, $z_{t-L+1}$:

$$
a_{t,h}=z_{t+h}-z_{t-L+1},\qquad h=1,\ldots,H.
$$

The diffusion target concatenates this latent component with the standardised
daily log return at each future date, giving a tensor of shape
`(batch, 30, 9)`. The latent history uses the same anchor, while the current
latent level, current return, and selected state features provide additional
conditioning.

The implementation also supports `z` (latent levels) and `dz` (successive latent
increments). The conversion helpers retain these alternatives; the configured
run below uses `adz`.


#### Encode the full surface chronology

Apply the frozen autoencoder in chronological batches and collect one latent
state per date. The checks verify both the number of states and their date
ordering before these states are supplied to the trajectory datasets.


In [ ]:
# PRECOMPUTE LATENT
DIFF_IVS_LOADER = DataLoader(DIFF_IVS, batch_size=512, shuffle=False,
                             drop_last=False, num_workers=4, pin_memory=True)
COORDS = DIFF_IVS_LOADER.dataset.coords.to(device, non_blocking=True)
Z_list = []
date_list = []
with torch.inference_mode():
    for batch in DIFF_IVS_LOADER:
        x_grid = batch["x_grid"].to(device, non_blocking=True)  # (B,1,T,K)
        m_grid = batch["m_grid"].to(device, non_blocking=True)  # (B,1,T,K)
        if AE_MODEL.arch=="RAE":
            z, _ = AE_MODEL(x_grid, m_grid, COORDS)
        elif AE_MODEL.arch=="VAE":
            mu, logvar, z, _ = AE_MODEL.forward_deterministic(x_grid, m_grid, COORDS)
        Z_list.append(z.detach().cpu().numpy().astype(np.float32))
        date_list.extend(batch["date"])
Z_FULL = np.concatenate(Z_list, axis=0).astype(np.float32)
if Z_FULL.shape[0] != len(DIFF_IVS.dates):
    raise RuntimeError(f"Latent/date mismatch: {Z_FULL.shape[0]} for {len(DIFF_IVS.dates)}.")
dates_full = [str(pd.Timestamp(d).date()) for d in DIFF_IVS.dates]
date_list = [str(pd.Timestamp(d).date()) for d in date_list]
if dates_full != date_list:
    raise RuntimeError("Date order mismatch between dataset and dataloader output.")

#### Validate origin and target boundaries

The split checker prints origin dates, first target dates, and final target
dates separately. Its assertions verify that each target follows its origin,
the complete horizon stays within the end date, and the required history exists.


In [ ]:
# Confirm Diff Split
def check_diff_split(ds, name, origin_start, target_end, H):
    origin_start = pd.Timestamp(origin_start)
    target_end = pd.Timestamp(target_end)

    idx = np.array(ds.valid_t_indices)
    origins = pd.Series([ds.dates[t] for t in idx])
    first_preds = pd.Series([ds.dates[t + 1] for t in idx])
    last_preds = pd.Series([ds.dates[t + H] for t in idx])

    print(f"\n{name}")
    print(f"n samples:        {len(idx)}")
    print(f"origin range:     {origins.min().date()} -> {origins.max().date()}")
    print(f"first pred range: {first_preds.min().date()} -> {first_preds.max().date()}")
    print(f"last pred range:  {last_preds.min().date()} -> {last_preds.max().date()}")

    """
    if name=="train":
        print(len(idx))
        assert len(idx)==5712
    elif name=="val":
        print(len(idx))
        assert len(idx)==221
    elif name=="test":
        print(len(idx))
        assert len(idx)==637
    """

    assert (origins >= origin_start).all()
    assert (last_preds <= target_end).all()
    assert (first_preds > origins).all()
    assert (last_preds >= first_preds).all()

    for t in idx:
        assert t - ds.num_surf_hist + 1 >= 0
        assert t + H < len(ds.dates)

#### Select windows and conditioning features

| Dataset | Earliest origin | Latest permitted final target |
| :-- | :-- | :-- |
| Training | 1 January 2000, subject to available history | 31 December 2022 |
| Validation diagnostic | 1 January 2022 | 31 December 2022 |
| Test | 1 January 2023 | 31 December 2025 |

`S_=21` and `H_=30` specify the window lengths. The selected conditioning vector
contains ten return-derived features and seventeen surface-state features.
Histories are drawn from the common full chronology, allowing early test
origins to use preceding observations while keeping future targets after the origin.


In [ ]:
# DIFFUSION SETTINGS
train_start, train_end = "2000-01-01", "2022-12-31"
val_start, val_end = "2022-01-01", "2022-12-31"
test_start, test_end = "2023-01-01", "2025-12-31"
S_ = 21; H_ = 30 # num_surf_hist; num_ret_hist; num_preds
ft_cols = ["ret_neg_ind", "ret_sq", "cumret_5d", "cumret_21d", "rv_5d", "rv_21d",
           "min_21d", "dsv_21d", "ret_skew_21d", "vov_21d_on_rv5",
           "avg_level_norm_log_tiv","atm_level_norm_log_tiv","skew_left_minus_right_norm_log_tiv",
           "curvature_wings_minus_atm_norm_log_tiv","term_slope_long_minus_short_atm_norm_log_tiv",
           "d5_avg_level_norm_log_tiv", "d5_atm_level_norm_log_tiv", "d5_skew_left_minus_right_norm_log_tiv",
           "d5_curvature_wings_minus_atm_norm_log_tiv", "d5_term_slope_long_minus_short_atm_norm_log_tiv",
           "descriptor_move_norm_5d_norm_log_tiv",
           "d21_avg_level_norm_log_tiv", "d21_atm_level_norm_log_tiv", "d21_skew_left_minus_right_norm_log_tiv",
           "d21_curvature_wings_minus_atm_norm_log_tiv", "d21_term_slope_long_minus_short_atm_norm_log_tiv",
           "descriptor_move_norm_21d_norm_log_tiv"]

DIFF_TRAIN.update_diff_params(num_surf_hist=S_,
                              feature_cols=ft_cols,
                              num_preds=H_,
                              origin_start_date=train_start,
                              target_end_date=train_end,
                              target_mode="adz",
                              precomputed_latents=Z_FULL)
check_diff_split(DIFF_TRAIN, "train", train_start, train_end, H_)
DIFF_TRAIN_LOADER = DataLoader(DIFF_TRAIN, batch_size=64, shuffle=True,
                               drop_last=False, num_workers=4, pin_memory=True)
DIFF_VAL.update_diff_params(num_surf_hist=S_,
                              feature_cols=ft_cols,
                              num_preds=H_,
                              origin_start_date=val_start,
                              target_end_date=val_end,
                              target_mode="adz",
                              precomputed_latents=Z_FULL)
check_diff_split(DIFF_VAL, "val", val_start, val_end, H_)
DIFF_VAL_LOADER = DataLoader(DIFF_VAL, batch_size=64, shuffle=False,
                             drop_last=False, num_workers=4, pin_memory=True)
DIFF_TEST.update_diff_params(num_surf_hist=S_,
                              feature_cols=ft_cols,
                              num_preds=H_,
                              origin_start_date=test_start,
                              target_end_date=test_end,
                              target_mode="adz",
                              precomputed_latents=Z_FULL)
check_diff_split(DIFF_TEST, "test", test_start, test_end, H_)
DIFF_TEST_LOADER = DataLoader(DIFF_TEST, batch_size=64, shuffle=False,
                              drop_last=False, num_workers=4, pin_memory=True)

### 5.2 Scaling, denoiser, and diffusion process

The following components handle batch preparation, latent scaling, transformer
conditioning, forward noising, and reverse sampling. Latent scaling is estimated
from the training loader; returns and state features have already been scaled by
the dataset.


#### Batch and schedule utilities

Prepare the joint target and conditioning dictionary, define linear and cosine
noise schedules, and broadcast timestep coefficients over path tensors. The
remaining helpers zero conditioning for the unconditional option and flatten
the conditioning tensors in a consistent key order for the mean gate.


In [ ]:
def prepare_diffusion_batch(batch: Dict[str, torch.Tensor], device: torch.device
                            ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    x0 = batch["target"].to(device).float()
    cond = {"z_first": batch["z_first"].to(device).float(),
            "z_hist": batch["z_hist"].to(device).float(),
            "anchored_dz_hist": batch["anchored_dz_hist"].to(device).float(),
            "dz_hist": batch["dz_hist"].to(device).float(),
            "z_last": batch["z_last"].to(device).float(),
            "ret_t": batch["ret_t"].to(device).float(),
            "feat_static": batch["feat_static"].to(device).float()}
    return x0, cond


def cosine_beta_schedule(timesteps: int, s: float = 0.008, zero_terminal_snr: bool = False) -> torch.Tensor:
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)

    alphas_cumprod = torch.cos(
        ((x / timesteps) + s) / (1 + s) * math.pi * 0.5
    ) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    if zero_terminal_snr:
        alphas_cumprod[-1] = 0.0

    betas = 1.0 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    if zero_terminal_snr:
        return betas.clamp(min=1e-8, max=1.0)
    else:
        return betas.clamp(min=1e-8, max=0.999)

def linear_beta_schedule(timesteps: int, beta_start: float = 1e-4, beta_end: float = 2e-2) -> torch.Tensor:
    return torch.linspace(beta_start, beta_end, timesteps).clamp(min=1e-8, max=0.999)


def make_unconditional_cond(cond: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: torch.zeros_like(v) for k, v in cond.items()}


def extract(a: torch.Tensor,t: torch.Tensor,x_shape: Tuple[int, ...]) -> torch.Tensor:
    if t.ndim != 1:
        raise ValueError(f"t must be 1D with shape (B,), got {tuple(t.shape)}.")
    B = t.shape[0]
    out = a.gather(dim=0, index=t)
    return out.reshape(B, *((1,) * (len(x_shape) - 1)))


def flatten_cond_for_lambda_gate(cond: Dict[str, torch.Tensor]) -> torch.Tensor:
    pieces = []
    for key in sorted(cond.keys()):
        value = cond[key]
        if not torch.is_tensor(value):
            continue
        if value.ndim == 0:
            continue
        B = value.shape[0]
        pieces.append(value.reshape(B, -1).float())
    if len(pieces) == 0:
        raise ValueError("No tensor entries found in cond.")
    return torch.cat(pieces, dim=-1)

#### Reversible latent-path scaling

`LatentPathStandardizer` keeps distinct statistics for latent levels, daily
increments, and anchored paths. It supports a shared scale or position-specific
anchored-path scales. The configured run uses shared (`global`) scaling.
Return channels pass through this object unchanged because the dataset has
already standardised them.


In [ ]:
class LatentPathStandardizer(nn.Module):
    def __init__(self,z_mean: torch.Tensor,z_std: torch.Tensor,
                 dz_mean: torch.Tensor,dz_std: torch.Tensor,
                 adz_mean: torch.Tensor,adz_std: torch.Tensor,
                 latent_dim: int,ret_dim: int = 1,eps: float = 1e-8,
                 horizon_normalization: str = "global"):
        super().__init__()
        self.latent_dim = int(latent_dim)
        self.ret_dim = int(ret_dim)
        self.horizon_normalization = horizon_normalization
        if self.horizon_normalization not in ("global", "horizon"):
            raise ValueError(f"Unknown horizon_normalization={horizon_normalization}.")
        self.register_buffer("z_mean", z_mean.float().view(-1))
        self.register_buffer("z_std", z_std.float().view(-1).clamp_min(eps))
        self.register_buffer("dz_mean", dz_mean.float().view(-1))
        self.register_buffer("dz_std", dz_std.float().view(-1).clamp_min(eps))
        adz_mean = adz_mean.float()
        adz_std = adz_std.float().clamp_min(eps)
        if self.horizon_normalization == "global":
            if adz_mean.ndim != 1 or adz_std.ndim != 1:
                raise ValueError("For horizon_normalization='global', adz_mean and adz_std must be 1D, got "
                                 f"{tuple(adz_mean.shape)} and {tuple(adz_std.shape)}.")
        if self.horizon_normalization == "horizon":
            if adz_mean.ndim != 1:
                raise ValueError("For horizon_normalization='horizon', adz_mean should still be "
                                 f"global 1D with shape (d,), got {tuple(adz_mean.shape)}.")
            if adz_std.ndim != 2 or adz_std.shape[-1] != self.latent_dim:
                raise ValueError("For horizon_normalization='horizon', adz_std must be "
                                 f"(L+H,{self.latent_dim}), got {tuple(adz_std.shape)}.")
        self.register_buffer("adz_mean", adz_mean)
        self.register_buffer("adz_std", adz_std)

    @staticmethod
    def _norm_last_dim(x: torch.Tensor,mean: torch.Tensor,std: torch.Tensor) -> torch.Tensor:
        shape = [1] * x.ndim
        shape[-1] = -1
        return (x - mean.view(*shape)) / std.view(*shape)

    @staticmethod
    def _denorm_last_dim(x: torch.Tensor,mean: torch.Tensor,std: torch.Tensor) -> torch.Tensor:
        shape = [1] * x.ndim
        shape[-1] = -1
        return x * std.view(*shape) + mean.view(*shape)

    @staticmethod
    def _norm_time_last_dim(x: torch.Tensor, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
        if x.ndim < 3:
            raise ValueError(f"Expected at least 3D tensor (...,T,d), got {tuple(x.shape)}.")
        T = x.shape[-2]
        d = x.shape[-1]
        if std.ndim != 2 or std.shape != (T, d):
            raise ValueError(f"std must be ({T},{d}) for x={tuple(x.shape)}, got {tuple(std.shape)}.")
        std_shape = [1] * x.ndim
        std_shape[-2] = T
        std_shape[-1] = d
        if mean.ndim == 1:
            if mean.shape[0] != d:
                raise ValueError(f"mean must have shape ({d},), got {tuple(mean.shape)}.")
            mean_shape = [1] * x.ndim
            mean_shape[-1] = d
        elif mean.ndim == 2:
            if mean.shape != (T, d):
                raise ValueError(f"2D mean must have shape ({T},{d}), got {tuple(mean.shape)}.")
            mean_shape = [1] * x.ndim
            mean_shape[-2] = T
            mean_shape[-1] = d
        else:
            raise ValueError(f"mean must be 1D or 2D, got {tuple(mean.shape)}.")

        return (x - mean.view(*mean_shape)) / std.view(*std_shape)

    @staticmethod
    def _denorm_time_last_dim(x: torch.Tensor, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
        if x.ndim < 3:
            raise ValueError(f"Expected at least 3D tensor (...,T,d), got {tuple(x.shape)}.")
        T = x.shape[-2]
        d = x.shape[-1]
        if std.ndim != 2 or std.shape != (T, d):
            raise ValueError(f"std must be ({T},{d}) for x={tuple(x.shape)}, got {tuple(std.shape)}.")
        std_shape = [1] * x.ndim
        std_shape[-2] = T
        std_shape[-1] = d
        if mean.ndim == 1:
            if mean.shape[0] != d:
                raise ValueError(f"mean must have shape ({d},), got {tuple(mean.shape)}.")
            mean_shape = [1] * x.ndim
            mean_shape[-1] = d
        elif mean.ndim == 2:
            if mean.shape != (T, d):
                raise ValueError(f"2D mean must have shape ({T},{d}), got {tuple(mean.shape)}.")
            mean_shape = [1] * x.ndim
            mean_shape[-2] = T
            mean_shape[-1] = d
        else:
            raise ValueError(f"mean must be 1D or 2D, got {tuple(mean.shape)}.")
        return x * std.view(*std_shape) + mean.view(*mean_shape)


    def _select_adz_std(self, x: torch.Tensor, adz_part: str) -> torch.Tensor:
        if self.horizon_normalization == "global":
            return self.adz_std
        if x.ndim < 3:
            raise ValueError(f"ADZ horizon normalisation expects x with shape (...,T,d), got {tuple(x.shape)}.")
        T = x.shape[-2]
        total_T = self.adz_std.shape[0]
        if T > total_T:
            raise ValueError(f"ADZ sequence length T={T} exceeds stored adz_std length {total_T}.")
        if adz_part == "history":
            return self.adz_std[:T]
        if adz_part == "future":
            return self.adz_std[-T:]
        if adz_part == "path":
            if T != total_T:
                raise ValueError(f"adz_part='path' requires T={total_T}, got T={T}.")
            return self.adz_std
        raise ValueError(
            f"Unknown adz_part={adz_part}. Expected 'history', 'future', or 'path'.")

    def normalize_latent(self, x: torch.Tensor, mode: str, adz_part: str=None) -> torch.Tensor:
        assert mode in ("z", "dz", "adz")
        if mode == "z": return self._norm_last_dim(x, self.z_mean, self.z_std)
        if mode == "dz": return self._norm_last_dim(x, self.dz_mean, self.dz_std)
        if mode == "adz":
            if self.horizon_normalization == "global":
                return self._norm_last_dim(x, self.adz_mean, self.adz_std)
            elif self.horizon_normalization == "horizon":
                assert adz_part is not None
                std = self._select_adz_std(x, adz_part=adz_part)
                return self._norm_time_last_dim(x, self.adz_mean, std)
            else:
                raise ValueError

    def denormalize_latent(self, x: torch.Tensor, mode: str, adz_part: str=None) -> torch.Tensor:
        assert mode in ("z", "dz", "adz")
        if mode == "z": return self._denorm_last_dim(x, self.z_mean, self.z_std)
        if mode == "dz": return self._denorm_last_dim(x, self.dz_mean, self.dz_std)
        if mode == "adz":
            if self.horizon_normalization == "global":
                return self._denorm_last_dim(x, self.adz_mean, self.adz_std)
            elif self.horizon_normalization == "horizon":
                assert adz_part is not None
                std = self._select_adz_std(x, adz_part=adz_part)
                return self._denorm_time_last_dim(x, self.adz_mean, std)
            else:
              raise ValueError

    def normalize_target(self,x0: torch.Tensor,target_mode: str) -> torch.Tensor:
        assert target_mode in ("z", "dz", "adz")
        x = x0.clone()
        latent = x[..., :self.latent_dim]
        if target_mode == "adz":
            x[..., :self.latent_dim] = self.normalize_latent(latent, mode="adz", adz_part="future")
        else:
            x[..., :self.latent_dim] = self.normalize_latent(latent, mode=target_mode)
        return x

    def denormalize_target(self,x0_norm: torch.Tensor,target_mode: str) -> torch.Tensor:
        assert target_mode in ("z", "dz", "adz")
        x = x0_norm.clone()
        latent = x[..., :self.latent_dim]
        if target_mode == "adz":
            x[..., :self.latent_dim] = self.denormalize_latent(latent, mode="adz", adz_part="future")
        else:
            x[..., :self.latent_dim] = self.denormalize_latent(latent, mode=target_mode)
        return x

    def normalize_cond(self,cond: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        out = dict(cond)
        for key in ["z_first", "z_hist", "z_last"]:
            if key in out:
                out[key] = self.normalize_latent(out[key], mode="z")
        if "dz_hist" in out:
            out["dz_hist"] = self.normalize_latent(out["dz_hist"], mode="dz")
        if "anchored_dz_hist" in out:
            out["anchored_dz_hist"] = self.normalize_latent(out["anchored_dz_hist"],mode="adz",adz_part="history")
        return out


#### Estimate scales from training windows

Accumulate first and second moments from the training loader. The fitting
routine centres latent levels and anchored paths while leaving daily increments
uncentred; the resulting statistics are saved with the diffusion checkpoint.


In [ ]:
@torch.no_grad()
def estimate_latent_path_standardizer(train_loader: DataLoader,device: torch.device,
                                      latent_dim: int,ret_dim: int = 1,eps: float = 1e-8,
                                      center_z: bool = True,center_dz: bool = False,
                                      center_adz: bool = False, horizon_normalization: str = "global"
                                      )-> LatentPathStandardizer:
    stats = {}
    adz_path_stats = None

    def init_if_needed(name: str, d: int, dtype: torch.dtype):
        if name not in stats:
            stats[name] = {"sum": torch.zeros(d, device=device, dtype=dtype),
                           "sumsq": torch.zeros(d, device=device, dtype=dtype), "n": 0}

    def update(name: str, x: torch.Tensor):
        x = x.reshape(-1, x.shape[-1])
        init_if_needed(name, x.shape[-1], x.dtype)
        stats[name]["sum"] += x.sum(dim=0)
        stats[name]["sumsq"] += (x ** 2).sum(dim=0)
        stats[name]["n"] += x.shape[0]

    def update_adz_path(x: torch.Tensor):
        nonlocal adz_path_stats
        if x.ndim != 3:
            raise ValueError(f"ADZ path must be (B,T,d), got {tuple(x.shape)}.")
        B, T, d = x.shape
        if d != latent_dim:
            raise ValueError(f"Expected latent_dim={latent_dim}, got d={d}.")
        if adz_path_stats is None:
            adz_path_stats = {"sum": torch.zeros(T, d, device=device, dtype=x.dtype),
                              "sumsq": torch.zeros(T, d, device=device, dtype=x.dtype), "n": 0}
        else:
            expected = tuple(adz_path_stats["sum"].shape)
            got = (T, d)
            if got != expected:
                raise ValueError(f"Inconsistent ADZ path shape. Got {got}, expected {expected}.")
        adz_path_stats["sum"] += x.sum(dim=0)
        adz_path_stats["sumsq"] += (x ** 2).sum(dim=0)
        adz_path_stats["n"] += B

    for batch in train_loader:
        z_hist = batch["z_hist"].to(device, non_blocking=True).float()
        z_fut = batch["z_fut"].to(device, non_blocking=True).float()
        dz_hist = batch["dz_hist"].to(device, non_blocking=True).float()
        dz_fut = batch["dz_fut"].to(device, non_blocking=True).float()
        adz_hist = batch["anchored_dz_hist"].to(device, non_blocking=True).float()
        adz_fut = batch["anchored_dz_fut"].to(device, non_blocking=True).float()
        update("z", z_hist)
        update("z", z_fut)
        update("dz", dz_hist)
        update("dz", dz_fut)
        update("adz", adz_hist)
        update("adz", adz_fut)
        # Position-wise ADZ scale statistics.
        if horizon_normalization == "horizon":
            if adz_hist.ndim != 3 or adz_fut.ndim != 3:
                raise ValueError(f"adz_hist and adz_fut must be 3D, got {tuple(adz_hist.shape)} and {tuple(adz_fut.shape)}.")
            adz_path = torch.cat([adz_hist, adz_fut], dim=1)
            update_adz_path(adz_path)

    def finish(name: str, center: bool):
        if name not in stats or stats[name]["n"] == 0:
            raise RuntimeError(f"No data collected for scaler '{name}'.")
        n = stats[name]["n"]
        mean = stats[name]["sum"] / n
        var = stats[name]["sumsq"] / n - mean ** 2
        std = var.clamp_min(eps).sqrt()
        if not center:
            mean = torch.zeros_like(mean)
        return mean.detach().cpu(), std.detach().cpu()

    z_mean, z_std = finish("z", center=center_z)
    dz_mean, dz_std = finish("dz", center=center_dz)
    adz_mean, adz_std_global = finish("adz", center=center_adz)
    if horizon_normalization == "global":
        adz_std = adz_std_global
    else:
        if adz_path_stats is None or adz_path_stats["n"] == 0:
            raise RuntimeError("No ADZ path-position stats collected.")
        n = adz_path_stats["n"]
        path_first_moment = adz_path_stats["sum"] / n        # (L+H,d)
        path_second_moment = adz_path_stats["sumsq"] / n     # (L+H,d)
        mu = adz_mean.to(device)
        adz_std = (path_second_moment - 2.0 * path_first_moment * mu.view(1, -1)
                   + mu.view(1, -1) ** 2).clamp_min(eps).sqrt()
        adz_std = adz_std.detach().cpu()

    scaler = LatentPathStandardizer(z_mean=z_mean,z_std=z_std,
                                    dz_mean=dz_mean,dz_std=dz_std,
                                    adz_mean=adz_mean,adz_std=adz_std,
                                    latent_dim=latent_dim,ret_dim=ret_dim,eps=eps,
                                    horizon_normalization=horizon_normalization)
    return scaler.to(device)

#### Time, state, and attention building blocks

Diffusion-step embeddings identify the current noise level. A state encoder
combines the current latent level, current return, and origin features into
one token. Residual self-attention blocks process that state together with the
history and noisy future tokens.


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim: int, use_mlp: bool = True):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

        if use_mlp:
            self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim),
                                     nn.SiLU(),nn.Linear(4 * dim, dim))
        else:
            self.mlp = None

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float()
                          / max(half - 1, 1))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)

        if self.mlp is not None:
            emb = self.mlp(emb)
        return emb


class ContinuationConditionEncoder(nn.Module):

    def __init__(self,latent_dim: int,ret_dim: int,feat_dim: int,d_model: int,dropout: float = 0.10):
        super().__init__()
        self.latent_dim = int(latent_dim)
        self.ret_dim = int(ret_dim)
        self.feat_dim = int(feat_dim)
        self.d_model = int(d_model)

        state_dim = self.latent_dim + self.ret_dim + self.feat_dim
        self.state_proj = nn.Sequential(nn.LayerNorm(state_dim),
                                        nn.Linear(state_dim, d_model),
                                        nn.SiLU(), nn.Dropout(dropout),
                                        nn.Linear(d_model, d_model))

    def forward(self, cond: Dict[str, torch.Tensor]) -> torch.Tensor:
        z_t = cond["z_last"]
        ret_t = cond["ret_t"]
        feat_static = cond["feat_static"]

        if z_t.ndim != 2 or z_t.shape[-1] != self.latent_dim:
            raise ValueError(f"z_last/z_t must be (B,{self.latent_dim}), got {tuple(z_t.shape)}.")
        if ret_t.ndim != 2 or ret_t.shape[-1] != self.ret_dim:
            raise ValueError(f"ret_t must be (B,{self.ret_dim}), got {tuple(ret_t.shape)}.")
        if feat_static.ndim != 2 or feat_static.shape[-1] != self.feat_dim:
            raise ValueError(f"feat_static must be (B,{self.feat_dim}), got {tuple(feat_static.shape)}.")

        state = torch.cat([z_t, ret_t, feat_static], dim=-1)
        return self.state_proj(state).unsqueeze(1)   # (B, 1, d_model)


class ContinuationSelfAttnBlock(nn.Module):

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.10, ffn_mult: int = 4):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ffn_mult * d_model),
                                nn.SiLU(), nn.Dropout(dropout),
                                nn.Linear(ffn_mult * d_model, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        a, _ = self.self_attn(h, h, h, need_weights=False)
        x = x + self.drop(a)

        h = self.norm2(x)
        x = x + self.drop(self.ff(h))
        return x


#### Transformer trajectory denoiser

The token sequence contains one state token, 21 latent-history tokens, and
30 noisy joint future tokens. Position and token-type embeddings distinguish
their roles, and a diffusion-time embedding is added throughout. The output
head reads only the future positions and predicts all nine target channels
at each horizon.


In [ ]:
class ContinuationTrajectoryDenoiser(nn.Module):

    def __init__(self,latent_dim: int = 8,ret_dim: int = 1,horizon: int = 30,target_mode: str = "adz",
                 max_path_history_len: int = 21,feat_dim: int = 10,d_model: int = 128,
                 n_layers: int = 4,n_heads: int = 4,dropout: float = 0.10,ffn_mult: int = 4,time_dim: int = 128):
        super().__init__()

        self.latent_dim = int(latent_dim)
        self.ret_dim = int(ret_dim)
        self.target_dim = self.latent_dim + self.ret_dim
        self.horizon = int(horizon)
        self.target_mode = str(target_mode)
        self.max_path_history_len = int(max_path_history_len)
        self.feat_dim = int(feat_dim)
        self.d_model = int(d_model)

        if target_mode == "z":
            self.path_key = "z_hist"
        elif target_mode == "dz":
            self.path_key = "dz_hist"
        elif target_mode == "adz":
            self.path_key = "anchored_dz_hist"
        else:
            raise ValueError(f"Unknown target_mode={target_mode}. Expected 'z', 'dz', or 'adz'.")

        self.cond_encoder = ContinuationConditionEncoder(latent_dim=self.latent_dim,
                                                         ret_dim=self.ret_dim, feat_dim=self.feat_dim,
                                                         d_model=d_model, dropout=dropout)

        self.hist_proj = nn.Linear(self.latent_dim, d_model)
        self.fut_proj = nn.Linear(self.target_dim, d_model)


        total_len = self.max_path_history_len + self.horizon
        self.pos_emb = nn.Embedding(total_len, d_model)
        self.type_emb = nn.Embedding(3, d_model)

        self.time_emb = TimeEmbedding(time_dim, use_mlp=True)
        self.time_proj = nn.Linear(time_dim, d_model)

        self.blocks = nn.ModuleList([ContinuationSelfAttnBlock(d_model=d_model, n_heads=n_heads,
                                                               dropout=dropout, ffn_mult=ffn_mult)
                                    for _ in range(n_layers)])

        self.out_norm = nn.LayerNorm(d_model)
        self.out_head = nn.Sequential(nn.Linear(d_model, d_model),
                                      nn.SiLU(), nn.Dropout(dropout),
                                      nn.Linear(d_model, self.target_dim))

    def _select_path(self, cond: Dict[str, torch.Tensor]) -> torch.Tensor:
        if self.path_key not in cond:
            raise KeyError(f"target_mode='{self.target_mode}' requires cond['{self.path_key}'], "
                           f"available keys: {list(cond.keys())}.")
        path = cond[self.path_key]
        if path.ndim != 3 or path.shape[-1] != self.latent_dim:
            raise ValueError(f"{self.path_key}: expected (B,L,{self.latent_dim}), got {tuple(path.shape)}.")
        if path.shape[1] < self.max_path_history_len:
            raise ValueError(f"{self.path_key}: history length {path.shape[1]} < max_path_history_len="
                             f"{self.max_path_history_len}.")
        if path.shape[1] > self.max_path_history_len:
            path = path[:, -self.max_path_history_len:, :]
        return path

    def forward(self,x_t: torch.Tensor,diffusion_t: torch.Tensor,
                cond: Dict[str, torch.Tensor],return_components: bool = False):
        B, H, D = x_t.shape
        if H != self.horizon:
            raise ValueError(f"Expected horizon={self.horizon}, got H={H}.")
        if D != self.target_dim:
            raise ValueError(f"Expected target_dim={self.target_dim}, got D={D}.")

        device = x_t.device
        L_hist = self.max_path_history_len

        # state token (1)
        state_tok = self.cond_encoder(cond)
        state_tok = state_tok + self.type_emb(
            torch.zeros(1, device=device, dtype=torch.long)
        )[None, :, :]

        # history tokens (2)
        path_hist = self._select_path(cond)
        hist_tok = self.hist_proj(path_hist)
        hist_pos = torch.arange(L_hist, device=device)
        hist_tok = hist_tok + self.pos_emb(hist_pos)[None, :, :]
        hist_tok = hist_tok + self.type_emb(
            torch.ones(1, device=device, dtype=torch.long)
        )[None, :, :]

        # future tokens (3)
        fut_tok = self.fut_proj(x_t)
        fut_pos = torch.arange(L_hist, L_hist + H, device=device)
        fut_tok = fut_tok + self.pos_emb(fut_pos)[None, :, :]
        fut_tok = fut_tok + self.type_emb(
            torch.full((1,), 2, device=device, dtype=torch.long)
        )[None, :, :]

        # broadcast diffusion-time embedding to every token (4)
        t_tok = self.time_proj(self.time_emb(diffusion_t)).unsqueeze(1)

        # full sequence (5)
        seq = torch.cat([state_tok, hist_tok, fut_tok], dim=1) + t_tok

        # model blocks (6)
        for block in self.blocks:
            seq = block(seq)

        # output (7)
        seq = self.out_norm(seq)
        fut_out = seq[:, 1 + L_hist:, :]
        model_out = self.out_head(fut_out)

        if return_components:
            return {"model_out": model_out, "memory": seq}
        return model_out


#### Forward noising and reverse samplers

For clean joint path $u_0$ and diffusion step $q$, the forward process is

$$
u_q=\sqrt{\bar\alpha_q}\,u_0+
\sqrt{1-\bar\alpha_q}\,\epsilon,
\qquad \epsilon\sim\mathcal N(0,I).
$$

The class converts among clean-path, noise, and $v$ predictions and provides
DDPM and DDIM reverse samplers. The selected objective is $v$-prediction,
$v_q=\sqrt{\bar\alpha_q}\epsilon-\sqrt{1-\bar\alpha_q}u_0$.
The denoising loss averages squared prediction error over origins, horizons,
and target channels, with latent and return diagnostics also reported separately.


In [ ]:
class GaussianDiffusionTrajectory(nn.Module):
    def __init__(self, denoiser: ContinuationTrajectoryDenoiser, timesteps: int = 1000,
                 beta_schedule: str = "cosine", loss_arch: str = "v"):
        super().__init__()

        assert loss_arch in ("eps", "v", "v-zeroSNR", "x0") or loss_arch.startswith("eps-clip-")
        self.denoiser = denoiser
        self.timesteps = timesteps
        self.loss_arch = loss_arch

        if beta_schedule == "cosine":
            if loss_arch == "v-zeroSNR":
                betas = cosine_beta_schedule(timesteps, zero_terminal_snr=True)
            else:
                betas = cosine_beta_schedule(timesteps)
        elif beta_schedule == "linear":
            betas = linear_beta_schedule(timesteps)
        else:
            raise ValueError(f"Unknown beta_schedule={beta_schedule}")

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))
        if self.loss_arch != "v-zeroSNR":
            self.register_buffer("sqrt_recip_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod))
            self.register_buffer("sqrt_recipm1_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod - 1.0))

        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.register_buffer("alphas_cumprod_prev", alphas_cumprod_prev)

        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.register_buffer("posterior_variance", posterior_variance)
        self.register_buffer("posterior_log_variance_clipped", torch.log(posterior_variance.clamp_min(1e-20)))
        self.register_buffer("posterior_mean_coef1", betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod))
        self.register_buffer("posterior_mean_coef2", (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod))

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0)
        return (extract(self.sqrt_alphas_cumprod, t, x0.shape) * x0 +
                extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape) * noise)

    def predict_v_target(self,x0: torch.Tensor,t: torch.Tensor,noise: torch.Tensor) -> torch.Tensor:
        return (extract(self.sqrt_alphas_cumprod, t, x0.shape) * noise -
                extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape) * x0)

    def predict_x0_from_eps(self,x_t: torch.Tensor,t: torch.Tensor,eps: torch.Tensor) -> torch.Tensor:
        return (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * eps)

    def predict_eps_from_x0(self,x_t: torch.Tensor,t: torch.Tensor,x0: torch.Tensor) -> torch.Tensor:
        return (x_t - extract(self.sqrt_alphas_cumprod, t, x_t.shape) * x0
               ) / extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape).clamp_min(1e-8)

    def predict_x0_from_v(self, x_t: torch.Tensor, t: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        return (extract(self.sqrt_alphas_cumprod, t, x_t.shape) * x_t -
                extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * v)

    def predict_eps_from_v(self, x_t: torch.Tensor, t: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        return (extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * x_t +
                extract(self.sqrt_alphas_cumprod, t, x_t.shape) * v)

    def model_predictions(self, x_t: torch.Tensor, t: torch.Tensor, cond: Dict[str, torch.Tensor],
                          clip_x0: Optional[float] = None, return_components: bool = False) -> Dict[str, torch.Tensor]:

        denoiser_out = self.denoiser(x_t=x_t,diffusion_t=t,cond=cond,return_components=return_components)
        if return_components:
            model_out = denoiser_out["model_out"]
        else:
            model_out = denoiser_out

        if self.loss_arch == "x0":
            x0_hat = model_out
            if clip_x0 is not None:
                x0_hat = x0_hat.clamp(-clip_x0, clip_x0)
            eps_hat = self.predict_eps_from_x0(x_t, t, x0_hat)
            v_hat = self.predict_v_target(x0_hat, t, eps_hat)

        elif self.loss_arch == "eps" or self.loss_arch.startswith("eps-clip-"):
            eps_hat = model_out
            x0_hat = self.predict_x0_from_eps(x_t, t, eps_hat)
            if clip_x0 is not None:
                x0_hat = x0_hat.clamp(-clip_x0, clip_x0)
                eps_hat = self.predict_eps_from_x0(x_t, t, x0_hat)
            v_hat = self.predict_v_target(x0_hat, t, eps_hat)

        elif self.loss_arch == "v" or self.loss_arch == "v-zeroSNR":
            v_hat = model_out
            x0_hat = self.predict_x0_from_v(x_t, t, v_hat)
            if clip_x0 is not None:
                x0_hat = x0_hat.clamp(-clip_x0, clip_x0)
                eps_hat = self.predict_eps_from_x0(x_t, t, x0_hat)
                v_hat = self.predict_v_target(x0_hat, t, eps_hat)
            else:
                eps_hat = self.predict_eps_from_v(x_t, t, v_hat)

        else:
            raise RuntimeError(f"Unhandled loss_arch={self.loss_arch}.")

        out = {"model_out": model_out,
               "x0_hat": x0_hat,
               "eps_hat": eps_hat,
               "v_hat": v_hat}
        if return_components:
            out["memory"] = denoiser_out["memory"]
        return out

    def training_loss(self, x0: torch.Tensor, cond: Dict[str, torch.Tensor],
                      t: Optional[torch.Tensor] = None, noise: Optional[torch.Tensor] = None
                      ) -> Dict[str, torch.Tensor]:
        if t is None:
            t = torch.randint(low=0, high=self.timesteps, size=(x0.shape[0],),
                              device=x0.device, dtype=torch.long)
        if noise is None:
            noise = torch.randn_like(x0)

        x_t = self.q_sample(x0, t, noise=noise)
        pred = self.model_predictions(x_t=x_t,t=t,cond=cond,clip_x0=None,return_components=False)

        if self.loss_arch == "x0":
            pred_hat = pred["model_out"]
            target = x0
        elif self.loss_arch == "eps" or self.loss_arch.startswith("eps-clip-"):
            pred_hat = pred["model_out"]
            target = noise
        elif self.loss_arch == "v" or self.loss_arch == "v-zeroSNR":
            pred_hat = pred["model_out"]
            target = self.predict_v_target(x0, t, noise)
        else:
            raise RuntimeError(f"Unhandled loss_arch={self.loss_arch}.")

        loss_tensor = (pred_hat - target) ** 2
        loss = loss_tensor.mean()
        x0_hat = pred["x0_hat"]
        x0_loss_tensor = (x0_hat - x0) ** 2
        x0_mse = x0_loss_tensor.mean()

        latent_dim = self.denoiser.latent_dim
        return {"loss": loss,
                "pred_mse": loss.detach(),
                "pred_mse_z": loss_tensor[:, :, :latent_dim].mean().detach(),
                "pred_mse_ret": loss_tensor[:, :, latent_dim:].mean().detach(),
                "x0_mse": x0_mse.detach(),
                "x0_mse_z": x0_loss_tensor[:, :, :latent_dim].mean().detach(),
                "x0_mse_ret": x0_loss_tensor[:, :, latent_dim:].mean().detach()}

    @torch.no_grad()
    def p_sample_ddpm(self,x_t: torch.Tensor,t: torch.Tensor,cond: Dict[str, torch.Tensor],
                      clip_x0: Optional[float] = None) -> torch.Tensor:
        pred = self.model_predictions(x_t=x_t,t=t,cond=cond,clip_x0=clip_x0)
        x0_hat = pred["x0_hat"]
        model_mean = (extract(self.posterior_mean_coef1, t, x_t.shape) * x0_hat +
                      extract(self.posterior_mean_coef2, t, x_t.shape) * x_t)
        log_var = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        noise = torch.randn_like(x_t)
        nonzero_mask = (t != 0).float().reshape(x_t.shape[0], *((1,) * (x_t.ndim - 1)))
        return model_mean + nonzero_mask * torch.exp(0.5 * log_var) * noise

    @torch.no_grad()
    def sample_ddpm(self,shape: Tuple[int, int, int],cond: Dict[str, torch.Tensor],
                    clip_x0: Optional[float] = None) -> torch.Tensor:
        device = next(self.parameters()).device
        B = shape[0]
        x = torch.randn(shape, device=device)

        if self.loss_arch.startswith("eps-clip-"):
            time_steps = self.timesteps - int(self.loss_arch.split("-")[-1])
        else:
            time_steps = self.timesteps
        for i in reversed(range(time_steps)):
            t = torch.full((B,), i, device=device, dtype=torch.long)
            x = self.p_sample_ddpm(x, t, cond, clip_x0=clip_x0)

        return x

    @torch.no_grad()
    def sample_ddim(self, shape: Tuple[int, int, int], cond: Dict[str, torch.Tensor],
                    sampling_steps: int = 100, eta: float = 0.0, clip_x0: Optional[float] = None) -> torch.Tensor:
        device = next(self.parameters()).device
        B = shape[0]
        x = torch.randn(shape, device=device)

        if self.loss_arch.startswith("eps-clip-"):
            time_steps = self.timesteps - 1 - int(self.loss_arch.split("-")[-1])
        else:
            time_steps = self.timesteps - 1
        times = torch.linspace(time_steps,0,steps=sampling_steps,device=device).long()
        times = torch.unique_consecutive(times)
        time_list = times.tolist()
        time_pairs = list(zip(time_list, time_list[1:] + [-1]))
        for t_now, t_next in time_pairs:
            t = torch.full((B,), t_now, device=device, dtype=torch.long)
            pred = self.model_predictions(x_t=x,t=t,cond=cond,clip_x0=clip_x0)
            x0_hat = pred["x0_hat"]
            eps_hat = pred["eps_hat"]
            if t_next < 0:
                x = x0_hat
                continue
            t_next_batch = torch.full((B,), t_next, device=device, dtype=torch.long)
            alpha_now = extract(self.alphas_cumprod, t, x.shape)
            alpha_next = extract(self.alphas_cumprod, t_next_batch, x.shape)
            sigma = (eta * torch.sqrt(((1.0 - alpha_next) / (1.0 - alpha_now)).clamp_min(0.0)) *
                     torch.sqrt((1.0 - alpha_now / alpha_next).clamp_min(0.0)))
            direction = torch.sqrt((1.0 - alpha_next - sigma ** 2).clamp_min(0.0)) * eps_hat
            noise = torch.randn_like(x)
            x = torch.sqrt(alpha_next) * x0_hat + direction + sigma * noise

        return x

    @torch.no_grad()
    def sample(self, shape: Tuple[int, int, int], cond: Dict[str, torch.Tensor],
               method: str = "ddim", sampling_steps: int = 100, eta: float = 0.0,
               clip_x0: Optional[float] = None) -> torch.Tensor:

        if method == "ddim":
            return self.sample_ddim(shape=shape,cond=cond,sampling_steps=sampling_steps,
                                    eta=eta,clip_x0=clip_x0)
        if method == "ddpm":
            return self.sample_ddpm(shape=shape,cond=cond,clip_x0=clip_x0)

        raise ValueError(f"Unknown sampling method={method}.")

#### Conditional blending weights

`CondLambdaGate` maps flattened conditioning information to one bounded weight
per horizon. Its final layer is initialised to a common weight of 0.75. The
gate is trained after the denoiser and is applied to latent ensemble means
during forecast-archive construction.


In [ ]:
class CondLambdaGate(nn.Module):

    def __init__(self, cond_dim: int, horizon: int, hidden_dim: int = 128,
                 max_lambda: float = 1.0, init_lambda: float = 0.75):
        super().__init__()
        self.horizon = int(horizon)
        self.max_lambda = float(max_lambda)

        self.net = nn.Sequential(nn.Linear(cond_dim, hidden_dim),
                                 nn.LayerNorm(hidden_dim), nn.SiLU(),
                                 nn.Linear(hidden_dim, hidden_dim),
                                 nn.LayerNorm(hidden_dim), nn.SiLU(),
                                 nn.Linear(hidden_dim, horizon))
        init_p = min(max(init_lambda / self.max_lambda, 1e-4), 1.0 - 1e-4)
        init_logit = math.log(init_p / (1.0 - init_p))
        final = self.net[-1]
        nn.init.zeros_(final.weight)
        nn.init.constant_(final.bias, init_logit)

    def forward(self, cond_flat: torch.Tensor) -> torch.Tensor:
        lam = self.max_lambda * torch.sigmoid(self.net(cond_flat))
        return lam.unsqueeze(-1)  # (B, H, 1)

### 5.3 Training helpers and conditional mean gate

The denoiser is first fitted to the joint trajectory objective. In the final
epoch, it is frozen and a smaller conditional gate is trained on latent
ensemble-mean error. The gate predicts one weight per origin and horizon,
blending the predicted latent mean with latent persistence. Section 6 shows
how that mean adjustment is applied to a sampled ensemble.


#### Denoiser updates and final-epoch gate fitting

The main epoch loop normalises paths and conditioning, samples noise levels,
and updates the denoiser. On the final epoch, the denoiser is frozen and the
gate is fitted for 15 passes through the training loader. Gate targets use an
eight-draw DDIM estimate of the latent mean, blended with latent persistence;
the gate objective is latent mean squared error.


In [ ]:
def train_one_epoch(diffusion: GaussianDiffusionTrajectory,train_loader: DataLoader,
                    optimizer: torch.optim.Optimizer,path_scaler: LatentPathStandardizer,
                    device: torch.device,amp_scaler: Optional[torch.cuda.amp.GradScaler] = None,
                    grad_clip: Optional[float] = 1.0,use_amp: bool = True,
                    target_mode: str = "adz",unconditional: bool = False,
                    last_epoch: bool=False) -> Dict[str, float]:

    diffusion.train()
    path_scaler.eval()

    totals = {"loss": 0.0,
              "pred_mse": 0.0,
              "pred_mse_z": 0.0,
              "pred_mse_ret": 0.0,
              "x0_mse": 0.0,
              "x0_mse_z": 0.0,
              "x0_mse_ret": 0.0}
    n_batches = 0
    amp_enabled = use_amp and device.type == "cuda"

    for batch in train_loader:
        x0_raw, cond_raw = prepare_diffusion_batch(batch, device)
        x0 = path_scaler.normalize_target(x0_raw, target_mode=target_mode)
        cond = path_scaler.normalize_cond(cond_raw)
        if unconditional:
            cond = make_unconditional_cond(cond)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            loss_dict = diffusion.training_loss(x0=x0, cond=cond)
            loss = loss_dict["loss"]
        if amp_scaler is not None and amp_enabled:
            amp_scaler.scale(loss).backward()
            if grad_clip is not None and grad_clip > 0:
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(diffusion.parameters(), grad_clip)
            amp_scaler.step(optimizer)
            amp_scaler.update()
        else:
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(diffusion.parameters(), grad_clip)
            optimizer.step()

        for k in totals:
            totals[k] += float(loss_dict[k].detach().cpu())
        n_batches += 1

    out =  {"train_loss": totals["loss"] / max(n_batches, 1),
            "train_pred_mse": totals["pred_mse"] / max(n_batches, 1),
            "train_pred_mse_z": totals["pred_mse_z"] / max(n_batches, 1),
            "train_pred_mse_ret": totals["pred_mse_ret"] / max(n_batches, 1),
            "train_x0_mse": totals["x0_mse"] / max(n_batches, 1),
            "train_x0_mse_z": totals["x0_mse_z"] / max(n_batches, 1),
            "train_x0_mse_ret": totals["x0_mse_ret"] / max(n_batches, 1)}

    # ============================================================
    # Final-epoch lambda-gate training
    # ============================================================
    if last_epoch:
        print("Training conditional lambda gate inside final epoch...")

        LAMBDA_GATE_EPOCHS = 15
        LAMBDA_GATE_LR = 3e-4
        LAMBDA_GATE_WEIGHT_DECAY = 1e-3
        LAMBDA_GATE_HIDDEN = 32
        LAMBDA_GATE_MAX_LAMBDA = 1.0
        LAMBDA_GATE_INIT_LAMBDA = 0.75
        LAMBDA_N_SAMPLES = 8
        LAMBDA_METHOD = "ddim"
        LAMBDA_SAMPLING_STEPS = 25
        LAMBDA_ETA = 0.0
        LAMBDA_CLIP_X0 = None

        latent_dim = int(path_scaler.z_mean.numel())
        for p in diffusion.parameters():
            p.requires_grad_(False)
        diffusion.eval()
        path_scaler.eval()

        lambda_gate = None
        lambda_optimizer = None
        lambda_totals = {
            "lambda_loss": 0.0,
            "lambda_raw_loss": 0.0,
            "lambda_pers_loss": 0.0,
            "lambda_r2_vs_pers": 0.0,
            "lambda_raw_r2_vs_pers": 0.0,
            "lambda_mean": 0.0,
            "lambda_low_share": 0.0,
            "lambda_high_share": 0.0,
        }
        n_lambda_batches = 0

        for lambda_epoch in range(1, LAMBDA_GATE_EPOCHS + 1):
            print(lambda_epoch, " / ", LAMBDA_GATE_EPOCHS)
            epoch_totals = {k: 0.0 for k in lambda_totals}
            epoch_batches = 0
            for batch in train_loader:
                x0_true_raw, cond_raw = prepare_diffusion_batch(batch, device)
                cond = path_scaler.normalize_cond(cond_raw)
                if unconditional:
                    cond = make_unconditional_cond(cond)
                cond_flat = flatten_cond_for_lambda_gate(cond)
                B = x0_true_raw.shape[0]
                H = x0_true_raw.shape[1]
                D = x0_true_raw.shape[2]

                if lambda_gate is None:
                    lambda_gate = CondLambdaGate(
                        cond_dim=cond_flat.shape[-1],
                        horizon=H,
                        hidden_dim=LAMBDA_GATE_HIDDEN,
                        max_lambda=LAMBDA_GATE_MAX_LAMBDA,
                        init_lambda=LAMBDA_GATE_INIT_LAMBDA,
                    ).to(device)
                    lambda_optimizer = torch.optim.AdamW(
                        lambda_gate.parameters(),
                        lr=LAMBDA_GATE_LR,
                        weight_decay=LAMBDA_GATE_WEIGHT_DECAY,
                    )
                lambda_gate.train()

                z_preds = []
                last_sample_out = None
                with torch.no_grad():
                    for _ in range(LAMBDA_N_SAMPLES):
                        sample_out = sample_future_from_batch(
                            diffusion=diffusion,
                            batch=batch,
                            path_scaler=path_scaler,
                            device=device,
                            latent_dim=latent_dim,
                            method=LAMBDA_METHOD,
                            sampling_steps=LAMBDA_SAMPLING_STEPS,
                            eta=LAMBDA_ETA,
                            unconditional=unconditional,
                            clip_x0=LAMBDA_CLIP_X0,
                            target_mode=target_mode,
                        )
                        z_preds.append(sample_out["z_pred"])
                        last_sample_out = sample_out
                    z_samples = torch.stack(z_preds, dim=0)  # (S,B,H,L)
                    z_mean = z_samples.mean(dim=0).detach()
                    z_true = last_sample_out["z_true"].detach()
                    z_persistence = last_sample_out["z_persistence"].detach()
                    raw_loss = F.mse_loss(z_mean, z_true)
                    pers_loss = F.mse_loss(z_persistence, z_true)

                lambda_hat = lambda_gate(cond_flat)  # (B,H,1)
                z_scaled = z_persistence + lambda_hat * (z_mean - z_persistence)
                lambda_loss = F.mse_loss(z_scaled, z_true)
                lambda_optimizer.zero_grad(set_to_none=True)
                lambda_loss.backward()
                lambda_optimizer.step()

                with torch.no_grad():
                    lambda_r2 = 1.0 - lambda_loss / torch.clamp(pers_loss, min=1e-12)
                    raw_r2 = 1.0 - raw_loss / torch.clamp(pers_loss, min=1e-12)
                    lambda_mean = lambda_hat.mean()
                    lambda_low_share = (lambda_hat < 0.25).float().mean()
                    lambda_high_share = (lambda_hat > 0.75).float().mean()

                metrics = {
                    "lambda_loss": float(lambda_loss.detach().cpu()),
                    "lambda_raw_loss": float(raw_loss.detach().cpu()),
                    "lambda_pers_loss": float(pers_loss.detach().cpu()),
                    "lambda_r2_vs_pers": float(lambda_r2.detach().cpu()),
                    "lambda_raw_r2_vs_pers": float(raw_r2.detach().cpu()),
                    "lambda_mean": float(lambda_mean.detach().cpu()),
                    "lambda_low_share": float(lambda_low_share.detach().cpu()),
                    "lambda_high_share": float(lambda_high_share.detach().cpu()),
                }

                for k in epoch_totals:
                    epoch_totals[k] += metrics[k]
                epoch_batches += 1

            epoch_avg = {
                k: epoch_totals[k] / max(epoch_batches, 1)
                for k in epoch_totals
            }

            print(
                f"Lambda epoch {lambda_epoch:02d}/{LAMBDA_GATE_EPOCHS} | "
                f"lambda_R2={epoch_avg['lambda_r2_vs_pers']:.4f} | "
                f"raw_R2={epoch_avg['lambda_raw_r2_vs_pers']:.4f} | "
                f"mean_lambda={epoch_avg['lambda_mean']:.3f} | "
                f"low={epoch_avg['lambda_low_share']:.3f} | "
                f"high={epoch_avg['lambda_high_share']:.3f}"
            )

            for k in lambda_totals:
                lambda_totals[k] = epoch_avg[k]

            n_lambda_batches = epoch_batches

        # Attach gate directly to diffusion
        diffusion.lambda_gate = lambda_gate
        diffusion.lambda_gate_target_mode = target_mode
        diffusion.lambda_gate_latent_dim = latent_dim
        diffusion.lambda_gate_uses_same_cond = True
        for k in lambda_totals:
            out[k] = lambda_totals[k]
        out["lambda_n_batches"] = n_lambda_batches


    return out


#### Repeatable denoising-loss evaluation

Evaluate fixed seeded noise/timestep draws, with repeated draws per batch, to
reduce variation in loss comparisons. This measures the denoising objective;
decoded forecast accuracy is assessed separately from generated samples.


In [ ]:
@torch.no_grad()
def eval_one_epoch(diffusion: GaussianDiffusionTrajectory,loader: DataLoader,
                   path_scaler: LatentPathStandardizer,
                   device: torch.device,use_amp: bool = True,
                   target_mode: str = "adz", unconditional: bool = False,
                   base_seed: int = 12345, n_repeats: int = 2, prefix: str = "eval") -> Dict[str, float]:

    diffusion.eval()
    path_scaler.eval()

    totals = {"loss": 0.0,
              "pred_mse": 0.0,
              "pred_mse_z": 0.0,
              "pred_mse_ret": 0.0,
              "x0_mse": 0.0,
              "x0_mse_z": 0.0,
              "x0_mse_ret": 0.0}
    n_batches = 0

    amp_enabled = use_amp and device.type == "cuda"
    for batch_idx, batch in enumerate(loader):
        x0_raw, cond_raw = prepare_diffusion_batch(batch, device)
        x0 = path_scaler.normalize_target(x0_raw, target_mode=target_mode)
        cond = path_scaler.normalize_cond(cond_raw)
        if unconditional:
            cond = make_unconditional_cond(cond)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            for r in range(n_repeats):
                gen = torch.Generator(device=device)
                gen.manual_seed(base_seed + 100_000 * r + batch_idx)
                t = torch.randint(low=0,high=diffusion.timesteps,size=(x0.shape[0],),
                                  device=device,generator=gen,dtype=torch.long)
                noise = torch.randn(x0.shape,device=device,dtype=x0.dtype,generator=gen)

                loss_dict = diffusion.training_loss(x0=x0,cond=cond,t=t,noise=noise)

                for k in totals:
                    totals[k] += float(loss_dict[k].detach().cpu())
                n_batches += 1

    return {f"{prefix}_loss": totals["loss"] / max(n_batches, 1),
            f"{prefix}_pred_mse": totals["pred_mse"] / max(n_batches, 1),
            f"{prefix}_pred_mse_z": totals["pred_mse_z"] / max(n_batches, 1),
            f"{prefix}_pred_mse_ret": totals["pred_mse_ret"] / max(n_batches, 1),
            f"{prefix}_x0_mse": totals["x0_mse"] / max(n_batches, 1),
            f"{prefix}_x0_mse_z": totals["x0_mse_z"] / max(n_batches, 1),
            f"{prefix}_x0_mse_ret": totals["x0_mse_ret"] / max(n_batches, 1)}


#### Generate a batch of latent futures

Sample a joint future, reverse latent scaling, and convert its latent component
back to levels. The returned dictionary includes predicted and true latent
paths, latent persistence, return paths, and target-space representations.
The gate-fitting loop uses these objects to compare latent means.


In [ ]:
@torch.no_grad()
def sample_future_from_batch(diffusion: GaussianDiffusionTrajectory,batch: Dict[str, torch.Tensor],
                             path_scaler: LatentPathStandardizer,device: torch.device,latent_dim: int,
                             method: str = "ddim",sampling_steps: int = 100,eta: float = 0.0,
                             unconditional: bool = False,clip_x0: Optional[float] = None,
                             target_mode: str = "adz") -> Dict[str, torch.Tensor]:
    diffusion.eval()
    path_scaler.eval()

    x0_true_raw, cond_raw = prepare_diffusion_batch(batch, device)
    cond = path_scaler.normalize_cond(cond_raw)
    if unconditional:
        cond = make_unconditional_cond(cond)

    B, H, D = x0_true_raw.shape
    x_sample_norm = diffusion.sample(shape=(B, H, D),cond=cond,method=method,
                                     sampling_steps=sampling_steps,eta=eta,clip_x0=clip_x0)
    x_sample = path_scaler.denormalize_target(x_sample_norm,target_mode=target_mode)

    pred_latent = x_sample[:, :, :latent_dim]
    true_latent = x0_true_raw[:, :, :latent_dim]
    pred_ret = x_sample[:, :, latent_dim:]
    true_ret = x0_true_raw[:, :, latent_dim:]
    z_first = cond_raw["z_first"]
    z_last = cond_raw["z_last"]

    if target_mode == "z":
        z_pred = pred_latent
        z_true = true_latent
    elif target_mode == "dz":
        z_pred = z_last[:, None, :] + torch.cumsum(pred_latent, dim=1)
        z_true = z_last[:, None, :] + torch.cumsum(true_latent, dim=1)
    elif target_mode == "adz":
        z_pred = z_first[:, None, :] + pred_latent
        z_true = z_first[:, None, :] + true_latent

    z_persistence = z_last[:, None, :].expand_as(z_true)

    out = {"z_pred": z_pred,
           "z_true": z_true,
           "z_persistence": z_persistence,
           "ret_pred": pred_ret,
           "ret_true": true_ret,
           "target_latent_pred": pred_latent,
           "target_latent_true": true_latent,
           "x0_pred": x_sample,
           "x0_true": x0_true_raw}

    if target_mode == "adz":
        out["adz_pred"] = pred_latent
        out["adz_true"] = true_latent

    return out

#### Configuration and model factory

The dataclass records the available architecture, diffusion, and optimisation. `build_diffusion_model` passes the selected settings
to the denoiser and diffusion wrapper. The explicit run configuration in
Section 5.4 overrides several dataclass defaults.


In [ ]:
@dataclass
class DiffusionTrainConfig:
    latent_dim: int = 8
    ret_dim: int = 1
    horizon: int = 30
    max_dz_history_len: int = 21

    target_mode: str = "adz"
    horizon_normalization: str = "global"
    unconditional: bool = False
    feat_dim: int = 10

    d_model: int = 128
    n_layers: int = 4
    n_heads: int = 4
    dropout: float = 0.10
    ffn_mult: int = 4
    time_dim: int = 128

    diffusion_steps: int = 1000
    beta_schedule: str = "cosine"
    loss_arch: str = "v"

    lr: float = 2e-4
    weight_decay: float = 1e-4
    epochs: int = 100
    grad_clip: float = 1.0
    use_amp: bool = True

    sample_eval_every: Optional[int] = 10
    sample_eval_batches: Optional[int] = 10
    sample_methods_eval: Tuple[str, ...] = ("ddim",) # ("ddim", "ddpm")
    ddim_steps_eval: int = 50
    ddim_eta_eval: float = 0.0
    ddpm_eval_every: int = 10_000
    clip_x0_eval: Optional[float] = None

    sample_eval_n_samples: int = 32
    sample_eval_horizons: Tuple[int, ...] = (1, 5, 10, 21, 30)

    save_path: Optional[str] = None


def build_diffusion_model(config: DiffusionTrainConfig) -> GaussianDiffusionTrajectory:

    denoiser = ContinuationTrajectoryDenoiser(latent_dim=config.latent_dim, ret_dim=config.ret_dim,
                                              horizon=config.horizon, target_mode=config.target_mode,
                                              max_path_history_len=config.max_dz_history_len,
                                              feat_dim=config.feat_dim, d_model=config.d_model,
                                              n_layers=config.n_layers, n_heads=config.n_heads,
                                              dropout=config.dropout, ffn_mult=config.ffn_mult,
                                              time_dim=config.time_dim)

    diffusion = GaussianDiffusionTrajectory(denoiser=denoiser,timesteps=config.diffusion_steps,
                                            beta_schedule=config.beta_schedule,
                                            loss_arch=config.loss_arch)

    return diffusion


#### Full diffusion fit and checkpoint contents

Estimate the path scaler, construct the model, and train with AdamW and a
cosine learning-rate schedule. Each saved checkpoint includes the model,
normalisation statistics, configuration, history, and any fitted mean gate.
An optional evaluation loader adds denoising-loss diagnostics.


In [ ]:
def fit_latent_trajectory_diffusion(train_loader: DataLoader,
                                    config: DiffusionTrainConfig,
                                    device: Optional[torch.device] = None,
                                    eval_loader: Optional[DataLoader] = None,
                                    ) -> Tuple[GaussianDiffusionTrajectory, LatentPathStandardizer, List[Dict[str, float]]]:

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print("Estimating latent path standardizer from training set...")
    path_scaler = estimate_latent_path_standardizer(train_loader=train_loader,device=device,
                                                    latent_dim=config.latent_dim,ret_dim=config.ret_dim,
                                                    center_z=True,center_dz=False,center_adz=True,
                                                    horizon_normalization=config.horizon_normalization
                                                    ).to(device)

    diffusion = build_diffusion_model(config).to(device)

    optimizer = torch.optim.AdamW(diffusion.parameters(),lr=config.lr,weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=config.epochs,eta_min=config.lr * 0.05)

    amp_scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp and device.type == "cuda")

    history = []
    for epoch in range(1, config.epochs + 1):
        if epoch < config.epochs:
            train_metrics = train_one_epoch(diffusion=diffusion,train_loader=train_loader,
                                            optimizer=optimizer,path_scaler=path_scaler,
                                            device=device,amp_scaler=amp_scaler,grad_clip=config.grad_clip,
                                            use_amp=config.use_amp,target_mode=config.target_mode,
                                            unconditional=config.unconditional,last_epoch=False)
        else:
            train_metrics = train_one_epoch(diffusion=diffusion,train_loader=train_loader,
                                            optimizer=optimizer,path_scaler=path_scaler,
                                            device=device,amp_scaler=amp_scaler,grad_clip=config.grad_clip,
                                            use_amp=config.use_amp,target_mode=config.target_mode,
                                            unconditional=config.unconditional,last_epoch=True)
        scheduler.step()

        row = {"epoch": epoch,**train_metrics,"lr": scheduler.get_last_lr()[0]}
        if eval_loader is not None:
            eval_metrics = eval_one_epoch(diffusion=diffusion,loader=eval_loader,
                                          path_scaler=path_scaler,
                                          device=device,use_amp=config.use_amp,
                                          target_mode=config.target_mode,
                                          unconditional=config.unconditional,
                                          prefix="eval")
            row.update(eval_metrics)
        history.append(row)

        msg = (f"Epoch {epoch:03d} | "
               f"train_loss={row['train_loss']:.6f} | "
               f"train_x0={row['train_x0_mse']:.6f} | "
               f"train_z={row['train_pred_mse_z']:.6f} | "
               f"train_ret={row['train_pred_mse_ret']:.6f} | "
               f"lr={row['lr']:.2e}")
        if "eval_loss" in row:
            msg += (f" | eval_loss={row['eval_loss']:.6f}"
                    f" | eval_x0={row['eval_x0_mse']:.6f}")
        if "lambda_r2_vs_pers" in row:
            msg += (
                f" | lambda_R2={row['lambda_r2_vs_pers']:.4f}"
                f" | raw_R2={row['lambda_raw_r2_vs_pers']:.4f}"
                f" | lambda_mean={row['lambda_mean']:.3f}"
                f" | lambda_low={row['lambda_low_share']:.3f}"
                f" | lambda_high={row['lambda_high_share']:.3f}"
            )
        print(msg)

        if (eval_loader is not None and config.sample_eval_every is not None
            and config.sample_eval_every > 0 and epoch % config.sample_eval_every == 0):
            print(
                f"Epoch {epoch:03d} | decoded sample evaluation should be run with "
                f"build_diffusion_eval_dump(...) / run_full_evaluation(...) "
                f"using checkpoint: {config.save_path}"
            )

        if config.save_path is not None:
            torch.save({"epoch": epoch, "model_state_dict": diffusion.state_dict(),
                        "path_z_mean": path_scaler.z_mean.detach().cpu(),
                        "path_z_std": path_scaler.z_std.detach().cpu(),
                        "path_dz_mean": path_scaler.dz_mean.detach().cpu(),
                        "path_dz_std": path_scaler.dz_std.detach().cpu(),
                        "path_adz_mean": path_scaler.adz_mean.detach().cpu(),
                        "path_adz_std": path_scaler.adz_std.detach().cpu(),
                        "lambda_gate_state_dict": (diffusion.lambda_gate.state_dict() if hasattr(diffusion, "lambda_gate")
                                                   and diffusion.lambda_gate is not None else None),
                        "lambda_gate_target_mode": (diffusion.lambda_gate_target_mode if
                                                    hasattr(diffusion, "lambda_gate_target_mode") else None),
                        "lambda_gate_latent_dim": (diffusion.lambda_gate_latent_dim if
                                                   hasattr(diffusion, "lambda_gate_latent_dim") else None),
                        "config": config, "history": history},
                config.save_path)

    return diffusion, path_scaler, history


#### Training-history plot

Plot the recorded training and available evaluation losses after fitting.


In [ ]:
def plot_loss_curves(history: List[Dict[str, float]]) -> None:
    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, train_loss, label="train")

    if len(history) > 0 and "eval_loss" in history[0]:
        eval_loss = [row["eval_loss"] for row in history]
        plt.plot(epochs, eval_loss, label="eval")

    plt.xlabel("Epoch")
    plt.ylabel("DDPM MSE")
    plt.title("Diffusion training loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

#### Restore a fitted model for inference

Rebuild the model from its saved configuration, restore any saved conditional
gate, load the parameter state and latent normalisation statistics, and freeze
the model for evaluation. The loader prints the checkpoint's core settings so
the sampled model can be identified in the session output.


In [ ]:
def load_diffusion_for_eval(checkpoint_path: str,device: Optional[torch.device] = None,
                            strict: bool = True
) -> Tuple[GaussianDiffusionTrajectory,
           LatentPathStandardizer, DiffusionTrainConfig,
           Dict[str, Any]]:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt = torch.load(checkpoint_path,map_location="cpu",weights_only=False)
    config = ckpt["config"]
    diffusion = build_diffusion_model(config)
    state = ckpt["model_state_dict"]
    cond_dim = state["lambda_gate.net.0.weight"].shape[1]
    hidden_dim = state["lambda_gate.net.0.weight"].shape[0]
    horizon = state["lambda_gate.net.6.weight"].shape[0]
    diffusion.lambda_gate = CondLambdaGate(cond_dim=cond_dim, horizon=horizon, hidden_dim=hidden_dim,
                                           max_lambda=1.0, init_lambda=0.75)
    diffusion.lambda_gate_target_mode = ckpt.get("lambda_gate_target_mode", None)
    diffusion.lambda_gate_latent_dim = ckpt.get("lambda_gate_latent_dim", None)
    diffusion.lambda_gate_uses_same_cond = ckpt.get("lambda_gate_uses_same_cond", True)
    diffusion.load_state_dict(state, strict=strict)

    diffusion.to(device)
    diffusion.eval()
    for d_p in diffusion.parameters():
        d_p.requires_grad_(False)

    path_scaler = LatentPathStandardizer(
        z_mean=ckpt["path_z_mean"],
        z_std=ckpt["path_z_std"],
        dz_mean=ckpt["path_dz_mean"],
        dz_std=ckpt["path_dz_std"],
        adz_mean=ckpt["path_adz_mean"],
        adz_std=ckpt["path_adz_std"],
        latent_dim=config.latent_dim,
        ret_dim=config.ret_dim,
    ).to(device)
    path_scaler.eval()
    for p_p in path_scaler.parameters():
        p_p.requires_grad_(False)

    print(f"Loaded checkpoint: {checkpoint_path}")
    print(f"Epoch: {ckpt.get('epoch', 'unknown')}")
    print(f"Device: {device}")
    print(
        f"horizon={config.horizon}, "
        f"latent_dim={config.latent_dim}, "
        f"ret_dim={config.ret_dim}, "
        f"target_mode={config.target_mode}, "
        f"diffusion_steps={config.diffusion_steps}, "
        f"schedule={config.beta_schedule}, "
        f"loss_arch={config.loss_arch}"
    )

    return diffusion, path_scaler, config, ckpt

### 5.4 Configured diffusion fit

| Setting | Value in this run |
| :-- | :-- |
| History / forecast horizon | 21 / 30 trading observations |
| Latent / return dimensions | 8 / 1 |
| Origin-date features | 27 |
| Transformer width / blocks / heads | 128 / 4 / 4 |
| Dropout | 0.10 |
| Diffusion steps / schedule | 500 / linear |
| Prediction target | $v$-prediction |
| Optimiser / learning rate | AdamW / $2\times10^{-4}$ |
| Training epochs | 100 |

The fit receives `eval_loader=None`. If the configured checkpoint exists, the
model is restored instead of fitted again. The final-epoch mean gate is part
of the saved model state.


In [ ]:
config = DiffusionTrainConfig(
    latent_dim=8,
    ret_dim=1,
    horizon=30,
    max_dz_history_len=21,

    target_mode="adz",
    horizon_normalization="global",
    unconditional=False,
    feat_dim=27,

    d_model=128,
    n_layers=4,
    n_heads=4,
    dropout=0.10,
    ffn_mult=4,
    time_dim=128,

    diffusion_steps=500,
    beta_schedule="linear",
    loss_arch="v",

    lr=2e-4,
    weight_decay=1e-4,
    epochs=100,
    grad_clip=1.0,
    use_amp=True,

    sample_eval_every=10_000,
    sample_eval_batches=10,
    sample_methods_eval=("ddim",),
    ddim_steps_eval=50,
    ddim_eta_eval=0,
    ddpm_eval_every=10_000,
    clip_x0_eval=None,

    sample_eval_n_samples=64,
    sample_eval_horizons=(1, 5, 10, 21, 30),
)
config.save_path = PATH_FINAL_DIFF_MODEL

if not os.path.exists(PATH_FINAL_DIFF_MODEL):
    diffusion, path_scaler, history = fit_latent_trajectory_diffusion(
        train_loader=DIFF_TRAIN_LOADER,
        config=config,
        eval_loader=None,
    )
    plot_loss_curves(history)
else:
    diffusion, path_scaler, _, _ = load_diffusion_for_eval(checkpoint_path=PATH_FINAL_DIFF_MODEL,
                                                           device=device, strict=True)
    print("Model loaded")

<a id="forecast-archive"></a>
## 6. Forecast archive

Sampling is separated from evaluation so that the same generated paths can be
used for several diagnostics. The archive retains latent targets, latent levels,
returns, decoded surfaces, observed surfaces and masks, dates, and normalisation
metadata. This also preserves the distinction between observed targets and
autoencoder reconstructions of those targets.


### 6.1 Decode paths and restore latent levels

The decoding helper flattens the leading origin/sample/horizon dimensions,
decodes manageable batches, and restores the original path shape. The target
conversion helper reverses the selected latent representation: add the first
history anchor for `adz`, cumulatively sum from the origin state for `dz`, or
retain levels for `z`.


In [ ]:
def decode_latent_trajectory(ae_model: Any, z: torch.Tensor, coords: torch.Tensor,
                             decode_batch_size: int, device: torch.device, amp_enabled: bool) -> torch.Tensor:
    """
    Input:
        z:
            (B, H, d_z) or (B, K, H, d_z)
    Output:
        y_hat:
            (B, H, P) or (B, K, H, P)  # P = T * K_grid is the number of IVS grid points.
    """
    leading_shape = z.shape[:-1]
    latent_dim = z.shape[-1]

    coords = coords.to(device=device, dtype=torch.float32, non_blocking=True)
    P = coords.shape[0]
    z_flat = z.reshape(-1, latent_dim).to(device=device, dtype=torch.float32, non_blocking=True)

    y_parts = []

    for start in range(0, z_flat.shape[0], decode_batch_size):
        z_chunk = z_flat[start:start + decode_batch_size]

        with autocast("cuda", enabled=(amp_enabled and device.type == "cuda")):
            y_chunk = ae_model.decode(coords, z_chunk)

        if y_chunk.ndim == 3 and y_chunk.shape[-1] == 1:
            y_chunk = y_chunk.squeeze(-1)

        if y_chunk.ndim != 2:
            raise ValueError(f"Unexpected decoder output shape: {tuple(y_chunk.shape)}")

        if y_chunk.shape[1] != P:
            raise ValueError(f"Decoder output {y_chunk.shape[1]} points, but coords {P} points.")
        y_chunk = y_chunk.detach().to(device="cpu", dtype=torch.float32)
        y_parts.append(y_chunk)

    y_flat = torch.cat(y_parts, dim=0)
    y = y_flat.reshape(*leading_shape, P)

    return y

def latent_target_to_z_path(
    latent_target: torch.Tensor,
    z_first: torch.Tensor,
    z_last: torch.Tensor,
    target_mode: str,
) -> torch.Tensor:

    if target_mode == "z":
        return latent_target

    if latent_target.ndim == 3:
        z_first_anchor = z_first[:, None, :]
        z_last_anchor = z_last[:, None, :]
        cumsum_dim = 1

    elif latent_target.ndim == 4:
        z_first_anchor = z_first[:, None, None, :]
        z_last_anchor = z_last[:, None, None, :]
        cumsum_dim = 2

    else:
        raise ValueError(
            f"Unexpected latent_target shape: {tuple(latent_target.shape)}"
        )

    if target_mode == "adz":
        return z_first_anchor + latent_target

    if target_mode == "dz":
        return z_last_anchor + torch.cumsum(latent_target, dim=cumsum_dim)

    raise ValueError(f"Unknown target_mode: {target_mode}")

### 6.2 Mean adjustment and archive construction

For each origin, generate the complete raw latent ensemble before applying the
conditional mean gate. If $\bar z_h$ is its mean, $z_t$ is latent persistence,
and $\lambda_{t,h}\in[0,1]$ is the gate output, each draw is adjusted as

$$
\widetilde z_h^{(s)}=
\underbrace{z_t+\lambda_{t,h}(\bar z_h-z_t)}_{\text{adjusted ensemble mean}}
+\underbrace{(z_h^{(s)}-\bar z_h)}_{\text{raw ensemble residual}}.
$$

The operation shifts the mean while retaining the raw ensemble residuals in
latent space. It does not rescale return draws. Each adjusted latent draw is
then decoded; because decoding is nonlinear, this latent-space property does
not imply an unchanged dispersion of decoded IV.

Observed future grids are stored separately from decoded true latent paths.
Surface tensors use half precision in the archive to reduce storage, while
latent states and returns use single precision. Dates, masks, coordinate order,
and inverse-normalisation statistics accompany the arrays.


In [ ]:
@torch.inference_mode()
def build_diffusion_eval_dump(checkpoint_path: str, ae_model, loader: torch.utils.data.DataLoader,
                              output_path: str, sampling_settings: List[Dict[str, Any]],
                              device: torch.device, use_amp: bool) -> Dict[str, Any]:

    random.seed(1234)
    np.random.seed(1234)
    torch.manual_seed(1234)
    torch.cuda.manual_seed_all(1234)

    diffusion, path_scaler, config, ckpt = load_diffusion_for_eval(
        checkpoint_path=checkpoint_path,
        device=device,
        strict=True,
    )

    coords = loader.dataset.coords.to(device=device, dtype=torch.float32, non_blocking=True)
    _r_mean, _r_std = loader.dataset.return_mean, loader.dataset.return_std
    target_mode = config.target_mode
    latent_dim = config.latent_dim
    ret_dim = config.ret_dim
    horizon = config.horizon
    unconditional = config.unconditional

    true_latent_target_parts = []
    true_ret_parts = []
    true_z_parts = []
    true_surface_parts = []
    z_first_parts = []
    z_last_parts = []
    observed_x_fut_parts = []
    observed_m_fut_parts = []
    observed_x_origin_parts = []
    observed_m_origin_parts = []
    date_origin_parts = []
    date_first_pred_parts = []
    date_last_pred_parts = []
    sample_latent_target_parts = {s["name"]: [] for s in sampling_settings}
    sample_ret_parts = {s["name"]: [] for s in sampling_settings}
    sample_z_parts = {s["name"]: [] for s in sampling_settings}
    sample_surface_parts = {s["name"]: [] for s in sampling_settings}

    n_origins = 0
    started_at = time.strftime("%Y-%m-%d %H:%M:%S")

    for batch_idx, batch in enumerate(loader):

        x0_true, cond_raw = prepare_diffusion_batch(batch, device)
        B, H, D = x0_true.shape
        assert H == horizon
        assert D == latent_dim + ret_dim

        true_latent_target = x0_true[:, :, :latent_dim]
        true_ret_norm = x0_true[:, :, latent_dim:]
        true_ret  = true_ret_norm * _r_std + _r_mean
        z_first = cond_raw["z_first"]
        z_last = cond_raw["z_last"]
        true_z = latent_target_to_z_path(
            latent_target=true_latent_target,
            z_first=z_first,
            z_last=z_last,
            target_mode=target_mode,
        )
        true_surface_norm = decode_latent_trajectory(ae_model=ae_model, z=true_z, coords=coords,
                                                     decode_batch_size=8192, device=device, amp_enabled=use_amp)

        true_latent_target_parts.append(true_latent_target.detach().cpu().float())
        true_ret_parts.append(true_ret.detach().cpu().float())
        true_z_parts.append(true_z.detach().cpu().float())
        true_surface_parts.append(true_surface_norm.detach().cpu().half())
        z_last_parts.append(z_last.detach().cpu().float())
        z_first_parts.append(z_first.detach().cpu().float())
        observed_x_fut = batch["x_fut_grid"].squeeze(2).reshape(B, H, -1)
        observed_x_fut_parts.append(observed_x_fut.detach().cpu().half())
        observed_m_fut = batch["m_fut_grid"].squeeze(2).reshape(B, H, -1)
        observed_m_fut_parts.append(observed_m_fut.detach().cpu().bool())
        observed_x_hist = batch["x_hist_grid"].squeeze(2).reshape(B, -1, coords.shape[0])
        observed_m_hist = batch["m_hist_grid"].squeeze(2).reshape(B, -1, coords.shape[0])
        observed_x_origin = observed_x_hist[:, -1, :]
        observed_m_origin = observed_m_hist[:, -1, :]
        observed_x_origin_parts.append(observed_x_origin.detach().cpu().half())
        observed_m_origin_parts.append(observed_m_origin.detach().cpu().bool())
        date_origin_parts.extend([str(x) for x in batch["date_origin"]])
        date_first_pred_parts.extend([str(x) for x in batch["date_first_pred"]])
        date_last_pred_parts.extend([str(x) for x in batch["date_last_pred"]])

        cond = path_scaler.normalize_cond(cond_raw)
        if unconditional:
            cond = make_unconditional_cond(cond)

        for setting in sampling_settings:

            name = setting["name"]
            method = setting["method"]
            n_samples = setting["n_samples"]
            sampling_steps = setting["sampling_steps"]
            eta = setting["eta"]
            clip_x0 = setting["clip_x0"]
            sample_chunk_size = setting["sample_chunk_size"]

            batch_ret_chunks = []
            batch_z_chunks = []

            # --------------------------------------------------------
            # 1) Sample all raw paths first.
            # --------------------------------------------------------
            for k_start in range(0, n_samples, sample_chunk_size):
                k_chunk = min(sample_chunk_size, n_samples - k_start)

                cond_chunk = {key: value.repeat_interleave(k_chunk, dim=0)
                              for key, value in cond.items()}

                with autocast(device_type=device.type,enabled=(use_amp and device.type == "cuda")):
                    x_sample_std = diffusion.sample(shape=(B * k_chunk, H, D), cond=cond_chunk,
                                                    method=method, sampling_steps=sampling_steps,
                                                    eta=eta, clip_x0=clip_x0)
                x_sample = path_scaler.denormalize_target(x_sample_std,target_mode=target_mode)
                x_sample = x_sample.reshape(B, k_chunk, H, D)

                pred_latent_target = x_sample[:, :, :, :latent_dim]
                pred_ret_norm = x_sample[:, :, :, latent_dim:]
                pred_ret = pred_ret_norm * _r_std + _r_mean
                pred_z = latent_target_to_z_path(
                    latent_target=pred_latent_target,
                    z_first=z_first,
                    z_last=z_last,
                    target_mode=target_mode,
                )
                batch_ret_chunks.append(pred_ret)
                batch_z_chunks.append(pred_z)

                del x_sample_std, x_sample, pred_latent_target, pred_ret_norm, pred_ret, pred_z, cond_chunk
                if device.type == "cuda":
                    torch.cuda.empty_cache()

            batch_z_raw = torch.cat(batch_z_chunks, dim=1)
            batch_ret = torch.cat(batch_ret_chunks, dim=1)

            # --------------------------------------------------------
            # 2) Apply lambda-gated mean + raw residuals in z-space.
            # --------------------------------------------------------
            use_lambda_mean_gate = True
            if use_lambda_mean_gate:
                if not hasattr(diffusion, "lambda_gate") or diffusion.lambda_gate is None:
                    raise ValueError("use_lambda_mean_gate=True, but diffusion.lambda_gate does not exist.")
                diffusion.lambda_gate.eval()
                cond_flat = flatten_cond_for_lambda_gate(cond)
                lambda_hat = diffusion.lambda_gate(cond_flat)
                if lambda_hat.shape != (B, H, 1):
                    raise ValueError(f"lambda_hat must have shape {(B, H, 1)}, got {tuple(lambda_hat.shape)}.")
                lambda_hat = lambda_hat.to(device=device, dtype=batch_z_raw.dtype)
                z_raw_mean = batch_z_raw.mean(dim=1)                 # (B,H,d)
                z_persistence = z_last[:, None, :].expand_as(z_raw_mean)
                z_lambda_mean = z_persistence + lambda_hat * (z_raw_mean - z_persistence)
                z_raw_residuals = batch_z_raw - z_raw_mean[:, None, :, :]
                batch_z = z_lambda_mean[:, None, :, :] + z_raw_residuals
            else:
                batch_z = batch_z_raw
            if target_mode == "adz":
                batch_latent_target = batch_z - z_first[:, None, None, :]
            elif target_mode == "z":
                batch_latent_target = batch_z
            elif target_mode == "dz":
                z_prev = torch.cat([z_last[:, None, None, :].expand(B, batch_z.shape[1], 1, latent_dim),
                                    batch_z[:, :, :-1, :],],dim=2,)
                batch_latent_target = batch_z - z_prev
            else:
                raise ValueError(f"Unknown target_mode: {target_mode}")

            # --------------------------------------------------------
            # 3) Decode adjusted z samples.
            # --------------------------------------------------------
            batch_surface_chunks = []

            for k_start in range(0, n_samples, sample_chunk_size):
                z_chunk = batch_z[:, k_start:k_start + sample_chunk_size]
                pred_surface_norm = decode_latent_trajectory(ae_model=ae_model, z=z_chunk, coords=coords,
                                                             decode_batch_size=512, device=device, amp_enabled=use_amp)
                batch_surface_chunks.append(pred_surface_norm.detach().cpu().half())

                del z_chunk, pred_surface_norm
                if device.type == "cuda":
                    torch.cuda.empty_cache()

            sample_latent_target_parts[name].append(batch_latent_target.detach().cpu().float())
            sample_ret_parts[name].append(batch_ret.detach().cpu().float())
            sample_z_parts[name].append(batch_z.detach().cpu().float())
            sample_surface_parts[name].append(torch.cat(batch_surface_chunks, dim=1))

            del batch_ret_chunks, batch_z_chunks
            del batch_z_raw, batch_ret, batch_z, batch_latent_target, batch_surface_chunks
            if device.type == "cuda":
                torch.cuda.empty_cache()

        n_origins += B
        print(f"batch {batch_idx + 1}: stored {n_origins} forecast origins")
        del x0_true, cond_raw, cond
        del true_latent_target, true_ret_norm, true_ret, true_z, true_surface_norm, z_first, z_last
        if device.type == "cuda":
            torch.cuda.empty_cache()

    observed_x_fut_all = torch.cat(observed_x_fut_parts, dim=0)
    observed_m_fut_all = torch.cat(observed_m_fut_parts, dim=0)
    observed_x_origin_all = torch.cat(observed_x_origin_parts, dim=0)
    observed_m_origin_all = torch.cat(observed_m_origin_parts, dim=0)
    lattice_mask = observed_m_fut_all[0, 0].clone()
    assert observed_m_fut_all.ndim == 3
    assert observed_m_origin_all.ndim == 2
    assert observed_x_fut_all.shape == observed_m_fut_all.shape
    assert observed_x_origin_all.shape == observed_m_origin_all.shape
    assert torch.equal(observed_m_fut_all,lattice_mask[None, None, :].expand_as(observed_m_fut_all))
    assert torch.equal(observed_m_origin_all,lattice_mask[None, :].expand_as(observed_m_origin_all))

    dump = {
        "created_at": started_at,
        "checkpoint_path": str(checkpoint_path),
        "config": dict(config.__dict__),
        "target_mode": target_mode,
        "latent_dim": latent_dim,
        "ret_dim": ret_dim,
        "horizon": horizon,
        "coords": coords.detach().cpu(),
        "sampling_settings": sampling_settings,
        "true": {
            "latent_target": torch.cat(true_latent_target_parts, dim=0),
            "ret": torch.cat(true_ret_parts, dim=0),
            "z": torch.cat(true_z_parts, dim=0),
            "surface_norm": torch.cat(true_surface_parts, dim=0),
            "z_first": torch.cat(z_first_parts, dim=0),
            "z_last": torch.cat(z_last_parts, dim=0),
        },
        "metadata": {
            "date_origin": date_origin_parts,
            "date_first_pred": date_first_pred_parts,
            "date_last_pred": date_last_pred_parts,
        },
        "path_standardizer": {
            "z_mean": ckpt["path_z_mean"].detach().cpu().float(),
            "z_std": ckpt["path_z_std"].detach().cpu().float(),
            "dz_mean": ckpt["path_dz_mean"].detach().cpu().float(),
            "dz_std": ckpt["path_dz_std"].detach().cpu().float(),
            "adz_mean": ckpt["path_adz_mean"].detach().cpu().float(),
            "adz_std": ckpt["path_adz_std"].detach().cpu().float(),
        },
        "surface_normalization": {
            "mean": loader.dataset.mean,
            "std": loader.dataset.std,
        },
        "samples": {},
        "observed": {
            "x_fut": observed_x_fut_all,
            "m_fut": observed_m_fut_all,
            "x_origin": observed_x_origin_all,
            "m_origin": observed_m_origin_all,
            "lattice_mask": lattice_mask,
        }
    }

    for setting in sampling_settings:
        name = setting["name"]
        dump["samples"][name] = {
            "setting": setting,
            "latent_target": torch.cat(sample_latent_target_parts[name], dim=0),
            "ret": torch.cat(sample_ret_parts[name], dim=0),
            "z": torch.cat(sample_z_parts[name], dim=0),
            "surface_norm": torch.cat(sample_surface_parts[name], dim=0),
        }
    assert dump["samples"][name]["surface_norm"].shape[0] == dump["observed"]["x_fut"].shape[0]
    assert dump["samples"][name]["surface_norm"].shape[2] == horizon
    assert dump["samples"][name]["surface_norm"].shape[3] == coords.shape[0]

    torch.save(dump, output_path)

    print(f"\nSaved evaluation dump to: {output_path}")
    print(f"target_mode: {target_mode}")
    print(f"N origins: {dump['true']['z'].shape[0]}")
    print(f"true latent_target: {tuple(dump['true']['latent_target'].shape)}")
    print(f"true ret:           {tuple(dump['true']['ret'].shape)}")
    print(f"true z:             {tuple(dump['true']['z'].shape)}")
    print(f"true surface_norm:  {tuple(dump['true']['surface_norm'].shape)}")
    print(f"observed x_fut:     {tuple(dump['observed']['x_fut'].shape)}")
    print(f"observed m_fut:     {tuple(dump['observed']['m_fut'].shape)}")
    print(f"observed x_origin:  {tuple(dump['observed']['x_origin'].shape)}")
    print(f"observed m_origin:  {tuple(dump['observed']['m_origin'].shape)}")
    print(f"lattice mask:       {tuple(dump['observed']['lattice_mask'].shape)}, "
          f"observed points = {int(dump['observed']['lattice_mask'].sum())}")

    for name, obj in dump["samples"].items():
        print(
            f"{name}: "
            f"latent_target {tuple(obj['latent_target'].shape)}, "
            f"ret {tuple(obj['ret'].shape)}, "
            f"z {tuple(obj['z'].shape)}, "
            f"surface_norm {tuple(obj['surface_norm'].shape)}"
        )

    return dump

### 6.3 Sampling settings

| Ensemble | Draws per origin | Reverse steps | $\eta$ | Sample chunk size |
| :-- | --: | --: | --: | --: |
| `ddpm` | 1,000 | 500 | 1.0 in settings | 16 |
| `ddim_50` | 1,000 | 50 | 0.0 | 32 |

DDPM uses its ancestral reverse process; the generic `eta` setting is used by
the DDIM sampler. DDIM with $\eta=0$ has a deterministic reverse path conditional
on its initial noise draw. The forecast archive is built only when `PATH_DUMP`
does not already exist.


In [ ]:
sampling_settings = [
    {
        "name": "ddpm",
        "method": "ddpm",
        "n_samples": 1000,
        "sampling_steps": 500,
        "eta": 1.0,
        "clip_x0": None,
        "sample_chunk_size": 16,
    },
    {
        "name": "ddim_50",
        "method": "ddim",
        "n_samples": 1000,
        "sampling_steps": 50,
        "eta": 0.0,
        "clip_x0": None,
        "sample_chunk_size": 32,
    },
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not os.path.exists(PATH_DUMP):
    build_diffusion_eval_dump(
        checkpoint_path=PATH_FINAL_DIFF_MODEL,
        ae_model=AE_MODEL,
        loader=DIFF_TEST_LOADER,
        output_path=PATH_DUMP,
        sampling_settings=sampling_settings,
        device=device,
        use_amp=True
    )
else:
    print("Dump Exists.")